> **Edición pública anonimizada.** Nombres y rutas comerciales se sustituyeron por etiquetas genéricas; no se incluyen bases ni modelos originales. Los resultados históricos se consultan en la tesis adaptada. La demostración pública usa datos y modelos sintéticos.


In [ ]:
from pathlib import Path
import os, sys

# Find the repository independently of the computer or clone location.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / ".tesis-portafolio").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Abre Jupyter desde la carpeta de este repositorio.")
os.chdir(REPO_ROOT / '04 Notebook Final')
sys.path.insert(0, str(REPO_ROOT / "01 Limpieza Código"))


<div style="border-left:5px solid #1565C0; background:#F5F9FF; padding:18px 22px; border-radius:4px;">
<div style="font-size:24px; font-weight:700; color:#0D47A1; line-height:1.25;">
Modelo vs. selección aleatoria
</div>
<div style="font-size:16px; color:#37474F; margin-top:4px;">
Análisis de los resultados de la simulación Monte Carlo
</div>
<div style="font-size:13px; color:#546E7A; margin-top:12px;">
Reporte analítico de resultados &nbsp;·&nbsp; No entrena modelos &nbsp;·&nbsp; No rehace el pipeline
</div>
</div>

<br>

### La pregunta que responde este reporte

> Si el equipo comercial solo puede contactar una fracción de los leads,
> **¿qué ventaja real obtiene al priorizar con el modelo en lugar de elegir leads al azar?**

Este notebook toma los resultados **ya producidos** por el experimento Monte Carlo del modelo final
(Sección 27 de `notebook_final_presentacion.ipynb`) y los somete a un análisis estadístico y
comercial en profundidad. No reconstruye el experimento.

### Contenido

| | Sección | Qué responde |
|---|---|---|
| **0** | Preparación | De dónde salen los datos y por qué son confiables |
| **1** | Executive Summary | Los hallazgos en cifras, para audiencia de negocio |
| **2** | Model vs. Random Selection | Comparación directa y P(azar ≥ modelo) |
| **3** | Monte Carlo Distribution | Forma completa de la distribución aleatoria |
| **4** | Incremental Value | Cuánto vale la diferencia en conversiones y esfuerzo |
| **5** | Commercial Interpretation | Qué cambia en el trabajo del equipo comercial |
| **6** | Practical Scenarios | Escenarios de capacidad y de carga individual |
| **7** | Risk and Uncertainty | Qué tan robusta es la ventaja y qué puede fallar |
| **8** | Qué demuestra el experimento | Interpretación conceptual — y qué NO demuestra |
| **9** | Limitations | Límites concretos de estos resultados |
| **10** | Conclusions | Respuestas directas a las ocho preguntas |

### Fuentes de datos — artefactos existentes, solo lectura

| Archivo | Contenido | Corrida |
|---|---|---|
| `modelo_final_montecarlo.csv` | Resultado del modelo y de la simulación aleatoria, 4 cortes | 4-ago |
| `modelo_final_umbrales.csv` | Puntos adicionales de la curva de ganancia (recall 50/60/70/80 %) | 4-ago |
| `modelo_final_metricas.csv` | PR-AUC, ROC-AUC, lift@10 %, recall@20 % (OOF, dos semillas) | 4-ago |
| `modelo_final_metricas_consolidadas.csv` | Métricas globales del modelo final | 4-ago |
| `modelo_final_matriz_confusion_metricas.csv` | Matriz de confusión en el umbral operativo | 4-ago |
| `operational_eval_conversion.csv` | Monte Carlo de 10 000 iteraciones × 32 configuraciones | 24-jul |

<div style="border-left:5px solid #F57C00; background:#FFF8F0; padding:14px 18px; border-radius:4px; margin-top:14px;">
<b style="color:#E65100;">Nota de trazabilidad</b><br>
En <code>qualified_run/results/</code> conviven artefactos de dos corridas distintas. Los archivos
<code>modelo_final_metricas_topk.csv</code>, <code>..._ranking.csv</code> y <code>..._umbral.csv</code>
son del <b>15 de julio</b> y corresponden a un universo anterior (20 304 leads / 625 conversiones).
<b>No se usan aquí</b>, para no mezclar corridas. Todo el análisis se apoya exclusivamente en los
artefactos del <b>4 de agosto</b> (22 656 leads / 739 conversiones).
</div>

<div style="border-left:5px solid #1565C0; background:#F5F9FF; padding:14px 18px; border-radius:4px; margin-top:10px;">
<b style="color:#0D47A1;">Qué se recalcula y por qué</b><br>
El experimento original guardó <b>solo el resumen</b> de la simulación (media e IC 95 %), no las
1 000 iteraciones individuales. Para analizar la distribución completa —percentiles, dispersión,
asimetría y sobre todo <b>P(azar ≥ modelo)</b>— se reconstruye la distribución aleatoria, que
<b>no requiere el modelo ni los datos</b>: depende únicamente del tamaño del universo, del número de
conversiones y de cuántos leads se contactan. La reconstrucción se <b>valida contra los valores
guardados</b> en la Sección 0.2.
</div>

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer este reporte (y qué significa cada palabra)</div>
<p style="margin:0 0 10px;">A lo largo del notebook aparecen cajas <b>moradas como esta</b>. Ninguna añade resultados nuevos: solo traducen a lenguaje corriente lo que dice la sección que las acompaña. Si algo no se entiende, la caja morada más cercana lo explica.</p><p style="margin:0 0 4px;"><b>La idea del reporte en tres frases:</b></p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;">Hay una lista enorme de <b>leads</b> (personas interesadas) y muy pocos terminan comprando.</li>
<li style="margin-bottom:6px;">El equipo comercial no puede llamar a todos: solo alcanza a contactar a una parte.</li>
<li style="margin-bottom:6px;">La pregunta es si <b>llamar en el orden que sugiere el modelo</b> consigue más ventas que <b>llamar sin ningún criterio</b> (al azar). Todo el reporte mide esa diferencia.</li>
</ul><p style="margin:14px 0 0;"><b>Vocabulario que se repite en todo el documento:</b></p><table style="border-collapse:collapse; width:100%; font-size:13px; margin-top:8px;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Término</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa aquí</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lead</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Una persona registrada como interesada. Es una fila de la base de datos.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conversión</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Un lead que sí terminó comprando. En este universo hay 739.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Universo H</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Los 22 656 leads sobre los que se evalúa el modelo: los que tienen informadas las cinco variables que el modelo necesita.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Tasa base</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El 3,26 % (739 de 22 656). Es la probabilidad de acertar tomando un lead cualquiera. Es el listón contra el que se compara todo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Modelo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Le pone una nota a cada lead y los ordena de mayor a menor probabilidad de comprar. No dice «este sí, este no»: dice «a estos llama primero».</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar / selección aleatoria</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La alternativa realista: trabajar la cartera sin priorizar, como si se sacaran leads de una bolsa.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Top 10 % (o «corte»)</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Contactar solo al 10 % mejor puntuado por el modelo: 2 266 leads. Los cortes son 5 %, 10 %, 20 % y 30 %.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Capacidad</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántos leads alcanza a contactar el equipo. Es el recurso escaso de todo el análisis.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Recall (% del negocio capturado)</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">De las 739 conversiones que existen, qué porcentaje cae dentro de los leads que sí se contactaron.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Precisión</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">De los leads contactados, qué porcentaje terminó comprando. Es «cuántas de mis llamadas sirvieron».</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lift</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas veces mejor que el azar. Lift de 3× = el modelo consigue el triple de conversiones que llamar sin criterio, con el mismo número de llamadas.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Monte Carlo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Repetir el sorteo miles de veces por computadora para ver todo lo que el azar <i>podría</i> haber dado, no solo un intento suelto.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Punto operativo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El nivel de capacidad que el proyecto eligió para operar: el más pequeño que alcanza a capturar al menos el 70 % de las conversiones.</td></tr>
</table>
</div>

---
## 0. Preparación

### 0.1 Carga de los resultados existentes

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import hypergeom, skew as sp_skew
from IPython.display import Markdown, HTML, display

# ── Rutas ────────────────────────────────────────────────────────────────
RESULTS_DIR = Path(f'{REPO_ROOT}/03 Prediccion/qualified_run/results')
BASE_OUT    = Path(f'{REPO_ROOT}/04 Notebook Final')
FIG_DIR     = BASE_OUT / 'figuras_montecarlo'     # salidas nuevas: no se toca nada existente
TAB_DIR     = BASE_OUT / 'tablas_montecarlo'
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Paleta (paleta categórica validada, modo claro) ──────────────────────
C_MODELO, C_AZAR, C_TERC = '#2a78d6', '#eb6834', '#1baf7a'
INK, INK2, GRIS = '#0b0b0b', '#52514e', '#c9c8c4'

mpl.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 10,
    'axes.edgecolor': GRIS, 'axes.labelcolor': INK, 'axes.titlecolor': INK,
    'axes.grid': True, 'grid.color': GRIS, 'grid.linewidth': 0.6, 'grid.alpha': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.color': INK2, 'ytick.color': INK2, 'legend.frameon': False,
    'axes.axisbelow': True, 'figure.facecolor': 'white',
})

# ── Presentación de tablas ───────────────────────────────────────────────
FUENTE = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif"
_CSS = [
    {'selector': '', 'props': [('border-collapse', 'collapse'), ('font-family', FUENTE),
                               ('font-size', '12.5px'), ('background', '#ffffff')]},
    {'selector': 'thead th', 'props': [('background', '#eceff3'), ('color', '#0b0b0b'),
                                       ('font-weight', '600'), ('text-align', 'right'),
                                       ('padding', '8px 12px'), ('white-space', 'nowrap'),
                                       ('border-bottom', '2px solid #c3c8cf')]},
    {'selector': 'thead th:first-child', 'props': [('text-align', 'left')]},
    {'selector': 'tbody td', 'props': [('text-align', 'right'), ('padding', '7px 12px'),
                                       ('color', '#1a1a1a'), ('white-space', 'nowrap'),
                                       ('border-bottom', '1px solid #edeef1')]},
    {'selector': 'tbody td:first-child', 'props': [('text-align', 'left'), ('font-weight', '500')]},
]

def tabla(df, titulo='', nota='', destacar=()):
    """Renderiza un DataFrame como tabla HTML limpia, con scroll horizontal si hace falta."""
    sty = df.style.hide(axis='index').set_table_styles(_CSS)
    for col in destacar:
        sty = sty.set_properties(subset=[col], **{'background-color': '#e8f1fc',
                                                  'font-weight': '600', 'color': '#0D47A1'})
    cab = (f'<div style="font:600 13.5px {FUENTE}; color:#0b0b0b; margin:16px 0 7px;">{titulo}</div>'
           if titulo else '')
    pie = (f'<div style="font:400 11.5px {FUENTE}; color:#6b6a66; margin:6px 0 4px; '
           f'max-width:900px; line-height:1.5;">{nota}</div>' if nota else '')
    display(HTML(cab + '<div style="overflow-x:auto;">' + sty.to_html() + '</div>' + pie))

def nota(texto, tipo='info'):
    """Bloque de lectura destacado. tipo: info | ok | alerta"""
    col = {'info': ('#1565C0', '#F5F9FF'), 'ok': ('#2E7D32', '#F4FBF5'),
           'alerta': ('#F57C00', '#FFF8F0')}[tipo]
    display(HTML(f'<div style="border-left:5px solid {col[0]}; background:{col[1]}; '
                 f'padding:12px 18px; border-radius:4px; margin:10px 0 16px; max-width:920px; '
                 f'font:400 13px {FUENTE}; color:#263238; line-height:1.6;">{texto}</div>'))

# ── Carga de artefactos ──────────────────────────────────────────────────
mc_src   = pd.read_csv(RESULTS_DIR / 'modelo_final_montecarlo.csv')
umbrales = pd.read_csv(RESULTS_DIR / 'modelo_final_umbrales.csv')
metricas = pd.read_csv(RESULTS_DIR / 'modelo_final_metricas.csv')
consolid = pd.read_csv(RESULTS_DIR / 'modelo_final_metricas_consolidadas.csv')
conf_mat = pd.read_csv(RESULTS_DIR / 'modelo_final_matriz_confusion_metricas.csv')

tabla(pd.DataFrame({
        'Corte'              : [f'Top {p}%' for p in mc_src['top_%']],
        'Leads contactados'  : mc_src.n_leads.map('{:,}'.format),
        'Conversiones modelo': mc_src.modelo_conversiones,
        'Recall modelo'      : (mc_src.modelo_recall * 100).map('{:.1f} %'.format),
        'Azar — media'       : mc_src.mc_media.map('{:.2f}'.format),
        'Azar — IC 95 %'     : [f'[{lo:.0f}, {hi:.0f}]' for lo, hi in
                                zip(mc_src.mc_ci95_lo, mc_src.mc_ci95_hi)],
        'Supera al azar'     : mc_src.supera_azar,
      }),
      titulo='Resultado guardado del experimento Monte Carlo — §27, 1 000 iteraciones',
      destacar=['Conversiones modelo'],
      nota='Fuente: <code>modelo_final_montecarlo.csv</code>. «***» indica que el modelo supera el '
           'límite superior del IC 95 % de la selección aleatoria.')

tabla(pd.DataFrame({
        'Recall objetivo' : (umbrales.recall_objetivo * 100).map('{:.0f} %'.format),
        'Umbral'          : umbrales.umbral.map('{:.4f}'.format),
        'Recall logrado'  : (umbrales.recall * 100).map('{:.1f} %'.format),
        'Precisión'       : (umbrales.precision * 100).map('{:.2f} %'.format),
        'Lift vs. base'   : umbrales.lift_vs_base.map('{:.2f}×'.format),
        'Leads'           : umbrales.leads.map('{:,}'.format),
        '% del universo'  : umbrales.pct_leads.map('{:.1f} %'.format),
      }),
      titulo='Umbrales operativos del modelo final',
      nota='Fuente: <code>modelo_final_umbrales.csv</code>. Cada fila es un punto real de la curva '
           'de ganancia; se usarán como niveles de capacidad adicionales en el análisis.')

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer las dos tablas de partida</div>
<p style="margin:0px 0 0;">Son los datos <b>tal como quedaron guardados</b> por el experimento original. Todavía no hay análisis: es el punto de partida.</p>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 1 — Resultado guardado del experimento Monte Carlo</p>
<p style="margin:4px 0 0;">Cada fila es un <b>corte</b>: contactar al 5, 10, 20 o 30 % mejor puntuado por el modelo.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Corte</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Qué porcentaje de la cartera se contacta, empezando por los mejor puntuados.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Leads contactados</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ese porcentaje traducido a número de llamadas.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conversiones modelo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas ventas cayeron dentro de esa lista. Dato observado.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Recall modelo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ese número como porcentaje de las 739 ventas del universo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — media</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Lo que habría dado en promedio elegir esa misma cantidad de leads sin criterio.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — IC 95 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El rango donde cae el azar 95 de cada 100 intentos.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Supera al azar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La marca <b>«***»</b> significa que el modelo quedó <b>por encima del extremo superior de ese rango</b>: mejor que el azar incluso con suerte.</td></tr>
</table>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 2 — Umbrales operativos del modelo final</p>
<p style="margin:4px 0 0;">Aquí cada fila es la respuesta a la pregunta inversa: <b>«¿cuántas llamadas necesito para capturar el X % del negocio?»</b>.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Recall objetivo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La meta fijada: capturar el 50, 60, 70 u 80 % de las ventas.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Umbral</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La <b>nota de corte</b>. El modelo puntúa a cada lead entre 0 y 1; un umbral de 0,0311 significa «llamo a todos los que saquen más de 0,0311». Es el número que convierte el ranking en una lista de trabajo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Recall logrado</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Lo que realmente se captura con ese umbral. Sale casi idéntico al objetivo (50,1 %, 60,1 %…) porque el umbral se buscó justo para eso.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Precisión</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">De los leads llamados, qué porcentaje compra. <b>Baja al bajar el umbral</b>, porque se van sumando leads cada vez peores.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lift vs. base</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas veces mejor es esa precisión que la tasa base del 3,26 %.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Leads / % del universo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas llamadas exige ese umbral, en absoluto y en porcentaje.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Qué mirar:</b> el intercambio entre las dos últimas columnas. Capturar el 80 % del negocio en lugar del 50 % casi <b>triplica</b> las llamadas (de 3 234 a 9 125) y baja el lift de 3,51× a 1,99×.</p>
</div>

### 0.2 Parámetros del universo y verificación de consistencia

El experimento se ejecutó sobre el **universo H** del modelo final: los leads que tienen informadas
las cinco variables requeridas. Sus dos parámetros —tamaño y número de conversiones— son los únicos
ingredientes necesarios para reconstruir la distribución aleatoria.

La celda siguiente **no asume** esos valores: los declara y luego **los verifica** contra los
archivos guardados. Si algún artefacto cambiara, las aserciones fallarían.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Qué se está haciendo en esta celda</div>
<p style="margin:0;">Antes de analizar nada, se comprueba que los números de partida sean los correctos. Se declaran los dos datos que definen el problema —<b>22 656 leads</b> en total y <b>739 conversiones</b> dentro de ellos— y se verifican contra los archivos guardados del experimento original.</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;"><b>¿Por qué solo hacen falta esos dos números?</b> Porque para saber qué logra el azar no hace falta saber quién es cada lead: solo cuántos hay y cuántos de ellos compran. Es como una bolsa con 22 656 bolas de las cuales 739 son premiadas.</li>
<li style="margin-bottom:6px;"><b>¿Qué es un <code>assert</code>?</b> Una alarma. Si algún archivo cambiara y los números dejaran de cuadrar, la celda se detendría con error en lugar de producir un reporte silenciosamente equivocado.</li>
<li style="margin-bottom:6px;"><b>Semilla (seed = 42):</b> el número que fija el «dado» de la computadora, para que quien repita el experimento obtenga exactamente los mismos resultados.</li>
</ul>
</div>

In [ ]:
# ── Parámetros del universo H (reportados por §21.4 del notebook final) ──
N_UNIVERSO = 22_656      # leads evaluados
N_CONV     = 739         # conversiones reales en ese universo
SEED       = 42          # semilla original del experimento
N_MC_ORIG  = 1_000       # iteraciones del experimento original
TASA_BASE  = N_CONV / N_UNIVERSO

chk = []
for _, r in mc_src.iterrows():
    tn = int(np.ceil(N_UNIVERSO * r['top_%'] / 100))     # regla original: int(np.ceil(n*k))
    esperado = tn * N_CONV / N_UNIVERSO                  # media exacta de la selección aleatoria
    chk.append({'corte': f"Top {int(r['top_%'])}%",
                'n_guardado': int(r['n_leads']), 'n_derivado': tn,
                'recall_guardado': r['modelo_recall'],
                'recall_derivado': round(r['modelo_conversiones'] / N_CONV, 3),
                'media_guardada': r['mc_media'], 'media_exacta': round(esperado, 2),
                'desvio': abs(r['mc_media'] - esperado) / esperado * 100})
chk = pd.DataFrame(chk)

assert (chk.n_guardado == chk.n_derivado).all(),           'N del universo inconsistente'
assert (chk.recall_guardado == chk.recall_derivado).all(), 'K (conversiones) inconsistente'
assert (chk.desvio < 1.5).all(),                           'La media aleatoria no reproduce'

tabla(pd.DataFrame({
        'Corte'                       : chk.corte,
        'Leads — guardado'            : chk.n_guardado.map('{:,}'.format),
        'Leads — derivado de N'       : chk.n_derivado.map('{:,}'.format),
        'Recall — guardado'           : chk.recall_guardado.map('{:.3f}'.format),
        'Recall — derivado de K'      : chk.recall_derivado.map('{:.3f}'.format),
        'Media azar — guardada'       : chk.media_guardada.map('{:.2f}'.format),
        'Media azar — exacta'         : chk.media_exacta.map('{:.2f}'.format),
        'Desvío'                      : chk.desvio.map('{:.2f} %'.format),
      }),
      titulo='Verificación de consistencia contra los archivos guardados')

nota(f'<b>Los tres controles pasan.</b> El universo es de <b>{N_UNIVERSO:,} leads</b> con '
     f'<b>{N_CONV} conversiones</b> reales, es decir una tasa base de <b>{TASA_BASE*100:.2f} %</b> '
     f'(1 de cada {1/TASA_BASE:.0f} leads convierte). Los números de leads por corte, el recall del '
     f'modelo y la media de la simulación reproducen exactamente lo guardado.', 'ok')

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer la tabla de verificación</div>
<p style="margin:0px 0 0;">Esta tabla <b>no aporta resultados</b>: es un control de calidad. Las columnas van en parejas — <b>lo que dice el archivo guardado</b> frente a <b>lo que debería dar el cálculo si los datos son los correctos</b>. Si una pareja no coincidiera, el notebook se detendría.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Pareja de columnas</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué comprueba</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Leads — guardado <br>vs. derivado de N</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El archivo dice «Top 10 % = 2 266 leads»; el cálculo comprueba que el 10 % de 22 656 son, en efecto, 2 266. Valida el <b>tamaño del universo</b>.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Recall — guardado <br>vs. derivado de K</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El archivo dice «recall 0,415»; el cálculo comprueba que 307 ÷ 739 = 0,415. Valida el <b>número de conversiones</b>.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Media azar — guardada <br>vs. exacta</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La media que dejó la simulación original frente a la media que da la fórmula matemática exacta.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Desvío</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La diferencia entre esas dos medias, en porcentaje. Debe ser mínima; el notebook exige que sea menor al 1,5 %.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Por qué no dan idénticas las medias:</b> la del archivo salió de sortear 1 000 veces, y mil sorteos siempre dejan un pequeño resto de ruido. La exacta no sortea nada. Que difieran en decimales es lo esperado; que difirieran mucho sería la señal de alarma.</p>
<p style="margin:10px 0 0;"><b>Qué mirar:</b> que las tres parejas coincidan y que la última columna esté cerca de cero. Eso confirma que se está analizando <b>la corrida correcta</b> (la del 4 de agosto: 22 656 leads y 739 conversiones) y no una mezcla de archivos de distintas fechas.</p>
</div>

### 0.3 Reconstrucción de la distribución aleatoria

El procedimiento original es: *tomar `n_leads` índices al azar sin reemplazo del universo y contar
cuántas conversiones caen dentro*. Ese conteo sigue **exactamente una distribución hipergeométrica**
`H(N = 22 656, K = 739, n = leads contactados)` — la identidad de cada lead es irrelevante, solo
importa cuántos positivos hay y cuántos se extraen.

Eso permite tres vías de cálculo, que se comparan a continuación:

1. **Réplica literal** del procedimiento original — muestreo sin reemplazo, 1 000 iteraciones, misma semilla.
2. **Simulación de alta resolución** — 200 000 iteraciones, para estimar percentiles y colas con precisión.
3. **Distribución exacta** — hipergeométrica analítica, sin error de muestreo. Es la referencia.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">La bolsa de bolas: qué es la distribución hipergeométrica</div>
<p style="margin:0 0 10px;">Imagine una bolsa con <b>22 656 bolas</b>, de las cuales <b>739 son premiadas</b> (las conversiones). Elegir leads al azar es exactamente lo mismo que meter la mano y sacar un puñado de bolas <b>sin devolverlas</b>. Si saca 2 266 bolas (el Top 10 %), en promedio le tocarán unas <b>74 premiadas</b>… pero unas veces saldrán 60 y otras 90. La <b>hipergeométrica</b> es simplemente la fórmula matemática que dice con qué frecuencia sale cada resultado posible en ese juego.</p><p style="margin:0 0 8px;">El notebook calcula lo mismo por <b>tres caminos distintos</b>, a propósito, para demostrar que no hay truco:</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;"><b>Vía 1 — repetir el experimento original tal cual:</b> 1 000 sorteos, la misma semilla. Sirve para confirmar que se está reproduciendo lo que ya se había hecho.</li>
<li style="margin-bottom:6px;"><b>Vía 2 — sortear muchísimas más veces (200 000):</b> con tantas repeticiones se ven bien los casos raros, es decir, qué tan lejos puede llegar el azar cuando tiene mucha suerte.</li>
<li style="margin-bottom:6px;"><b>Vía 3 — la fórmula exacta:</b> no sortea nada, calcula la respuesta directamente. Es la referencia contra la que se comparan las otras dos.</li>
</ul><p style="margin:10px 0 0;">Si los tres caminos coinciden —y coinciden—, los resultados del azar son confiables y no dependen de haber tenido suerte al simular.</p>
</div>

In [ ]:
CORTES  = mc_src['n_leads'].astype(int).tolist()
N_MC_HD = 200_000

# ── Vía 1: réplica literal del procedimiento original ────────────────────
# La POSICIÓN de los positivos es irrelevante (el conteo solo depende de cuántos se
# extraen), por eso se coloca con una semilla independiente de la del muestreo, para
# no inducir colisiones entre ambas secuencias del generador.
_rng_lab = np.random.default_rng(20_240_804)
y_lab = np.zeros(N_UNIVERSO, dtype=np.int8)
y_lab[_rng_lab.choice(N_UNIVERSO, N_CONV, replace=False)] = 1

rng_rep  = np.random.default_rng(SEED)
rep_1000 = {tn: np.array([y_lab[rng_rep.choice(N_UNIVERSO, tn, replace=False)].sum()
                          for _ in range(N_MC_ORIG)]) for tn in CORTES}

# ── Vía 2: simulación de alta resolución (mismo modelo probabilístico) ───
rng_hd = np.random.default_rng(SEED)
dist = {tn: rng_hd.hypergeometric(N_CONV, N_UNIVERSO - N_CONV, tn, size=N_MC_HD) for tn in CORTES}

# ── Vía 3: distribución exacta ───────────────────────────────────────────
exact = {tn: hypergeom(N_UNIVERSO, N_CONV, tn) for tn in CORTES}

cmp_rows = []
for _, r in mc_src.iterrows():
    tn = int(r['n_leads']); d, rp, hg = dist[tn], rep_1000[tn], exact[tn]
    cmp_rows.append({
        'Corte'                : f"Top {int(r['top_%'])}%",
        'Media — guardada'     : f"{r['mc_media']:.2f}",
        'Media — réplica'      : f'{rp.mean():.2f}',
        'Media — sim. 200k'    : f'{d.mean():.2f}',
        'Media — exacta'       : f'{hg.mean():.2f}',
        'IC 95 % — guardado'   : f"[{r['mc_ci95_lo']:.0f}, {r['mc_ci95_hi']:.0f}]",
        'IC 95 % — réplica'    : f'[{np.percentile(rp, 2.5):.0f}, {np.percentile(rp, 97.5):.0f}]',
        'IC 95 % — sim. 200k'  : f'[{np.percentile(d, 2.5):.0f}, {np.percentile(d, 97.5):.0f}]',
        'IC 95 % — exacto'     : f'[{hg.ppf(0.025):.0f}, {hg.ppf(0.975):.0f}]',
    })
tabla(pd.DataFrame(cmp_rows),
      titulo='Validación de la reconstrucción — tres vías de cálculo frente al valor guardado',
      destacar=['Media — guardada', 'IC 95 % — guardado'])

nota('<b>La reconstrucción es fiel.</b> Las cuatro medias coinciden hasta la primera decimal y los '
     'intervalos difieren a lo sumo en una conversión; las diferencias son el ruido de muestreo del '
     'experimento original, que usó 1 000 iteraciones. A partir de aquí se trabaja con la simulación '
     'de 200 000 iteraciones y con la distribución exacta, que permiten medir las colas.', 'ok')

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer la tabla de las tres vías</div>
<p style="margin:0px 0 0;">El experimento original guardó solo un resumen del azar, no las 1 000 simulaciones. Para poder analizarlo en profundidad hay que <b>reconstruir</b> esa distribución, y esta tabla demuestra que la reconstrucción es fiel. Cada fila es un corte; las columnas son <b>cuatro maneras de calcular lo mismo</b>, puestas lado a lado para compararlas.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">De dónde sale el número</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>… — guardada / guardado</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El valor que dejó escrito el experimento original. <b>Es la referencia a igualar</b> (va resaltado).</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>… — réplica</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Repetir el procedimiento original tal cual: 1 000 sorteos con la misma semilla. Comprueba que se entendió bien qué se hizo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>… — sim. 200k</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El mismo sorteo, pero 200 000 veces. Al repetir 200 veces más, el resultado es mucho más estable.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>… — exacta / exacto</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La fórmula matemática, sin sortear. No tiene ruido: <b>es la verdad del problema</b>.</td></tr>
</table>
<p style="margin:14px 0 0;">Las primeras cuatro columnas comparan la <b>media</b> (el resultado típico del azar) y las cuatro siguientes el <b>IC 95 %</b> (el rango donde cae el azar 95 de cada 100 veces).</p>
<p style="margin:10px 0 0;"><b>Qué mirar:</b> que las cuatro medias coincidan hasta la primera decimal y que los cuatro intervalos difieran como mucho en una conversión. Coinciden — por eso el resto del notebook puede trabajar con las 200 000 simulaciones y con la fórmula exacta, que sí permiten medir los casos extremos.</p>
</div>

### 0.4 Tabla maestra

Se construye una única tabla que consolida, para cada nivel de capacidad comercial, el resultado
observado del modelo y la distribución completa del azar. **Todas las secciones siguientes leen de
esta tabla**, de modo que ninguna cifra del reporte se escribe a mano.

A los 4 cortes del experimento se añaden **4 niveles de capacidad adicionales** recuperados de
`modelo_final_umbrales.csv`. No son escenarios inventados: son puntos reales de la curva de ganancia
del mismo modelo sobre el mismo universo, ya calculados. Para ellos la referencia aleatoria se
calcula con la misma distribución hipergeométrica.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Qué es la «tabla maestra» y por qué importa</div>
<p style="margin:0;">Es una sola tabla que reúne, para cada nivel de capacidad (cuántos leads se pueden contactar), <b>lo que logró el modelo</b> y <b>todo lo que habría logrado el azar</b>. Desde aquí hasta el final del notebook, <b>ninguna cifra se escribe a mano</b>: todas las tablas, gráficos y párrafos leen de esta tabla. Si un dato de origen cambiara, todo el reporte cambiaría solo, sin riesgo de que quede un número viejo perdido en el texto.</p><p style="margin:10px 0 0;">Los <b>ocho niveles</b> de capacidad no son inventados: cuatro vienen del experimento Monte Carlo (Top 5, 10, 20 y 30 %) y cuatro son umbrales reales del modelo (los que alcanzan 50, 60, 70 y 80 % de recall).</p>
</div>

In [ ]:
# ── Puntos del modelo: 4 del Monte Carlo + 4 de la tabla de umbrales ─────
puntos  = [{'n_leads': int(r['n_leads']), 'conv_modelo': int(r['modelo_conversiones']),
            'origen': 'Monte Carlo §27'} for _, r in mc_src.iterrows()]
puntos += [{'n_leads': int(r['leads']), 'conv_modelo': int(round(r['recall'] * N_CONV)),
            'origen': f"Umbral recall≥{r['recall_objetivo']:.0%}"} for _, r in umbrales.iterrows()]
grid = pd.DataFrame(puntos).sort_values('n_leads').reset_index(drop=True)

filas = []
for _, r in grid.iterrows():
    tn, m = int(r.n_leads), int(r.conv_modelo)
    hg = hypergeom(N_UNIVERSO, N_CONV, tn)
    d  = dist.get(tn)
    if d is None:                                    # cortes añadidos: se simulan igual
        d = np.random.default_rng(SEED + tn).hypergeometric(
                N_CONV, N_UNIVERSO - N_CONV, tn, size=N_MC_HD)
        dist[tn], exact[tn] = d, hg
    mu, sd = hg.mean(), hg.std()
    filas.append({
        'origen': r.origen, 'leads': tn, 'pct_universo': round(tn / N_UNIVERSO * 100, 1),
        # modelo (observado)
        'conv_modelo': m, 'recall_modelo': m / N_CONV, 'precision_modelo': m / tn,
        # azar (distribución completa)
        'azar_media': mu, 'azar_mediana': float(np.median(d)), 'azar_sd': sd,
        'azar_p2_5': float(np.percentile(d, 2.5)), 'azar_p97_5': float(np.percentile(d, 97.5)),
        'azar_min': int(d.min()), 'azar_max': int(d.max()),
        'azar_asimetria': float(sp_skew(d)), 'precision_azar': mu / tn,
        # comparación
        'delta_conv': m - mu, 'incremento_pct': (m - mu) / mu * 100, 'lift': m / mu,
        'z_modelo': (m - mu) / sd, 'p_azar_ge_modelo': float(hg.sf(m - 1)),
        'p_emp_azar_ge_mod': float((d >= m).mean()),
        # eficiencia operativa
        'conv_x100_modelo': m / tn * 100, 'conv_x100_azar': mu / tn * 100,
        'contactos_x_conv_modelo': tn / m, 'contactos_x_conv_azar': tn / mu,
        'improductivos_modelo': tn - m, 'improductivos_azar': tn - mu,
    })
res = pd.DataFrame(filas)
res.to_csv(TAB_DIR / 'tabla_maestra_montecarlo.csv', index=False)

tabla(pd.DataFrame({
        'Origen'          : res.origen,
        'Leads'           : res.leads.map('{:,}'.format),
        '% universo'      : res.pct_universo.map('{:.1f} %'.format),
        'Conv. modelo'    : res.conv_modelo,
        'Recall'          : (res.recall_modelo * 100).map('{:.1f} %'.format),
        'Azar — media'    : res.azar_media.map('{:.1f}'.format),
        'Azar — sd'       : res.azar_sd.map('{:.1f}'.format),
        'Azar — IC 95 %'  : [f'[{lo:.0f}, {hi:.0f}]' for lo, hi in zip(res.azar_p2_5, res.azar_p97_5)],
        'Δ conversiones'  : res.delta_conv.map('+{:.0f}'.format),
        'Lift'            : res.lift.map('{:.2f}×'.format),
        'Distancia (σ)'   : res.z_modelo.map('{:.1f}'.format),
      }),
      titulo='Tabla maestra — modelo observado vs. distribución del azar, por nivel de capacidad',
      destacar=['Conv. modelo', 'Lift'],
      nota='Guardada en <code>tablas_montecarlo/tabla_maestra_montecarlo.csv</code>. '
           'Las filas «Umbral recall≥X %» son puntos reales de la curva de ganancia, no extrapolaciones.')

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer la tabla maestra</div>
<p style="margin:0 0 4px;">Cada fila es <b>un escenario de capacidad</b>: «si el equipo puede hacer <i>tantas</i> llamadas, esto consigue con el modelo y esto conseguiría sin él». Están ordenadas de menor a mayor capacidad.</p>
<p style="margin:14px 0 0; font-weight:700; color:#7B1FA2;">Qué dice cada columna</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Origen</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">De dónde sale la fila. <b>«Monte Carlo §27»</b> = uno de los cuatro cortes del experimento original (Top 5, 10, 20 y 30 %). <b>«Umbral recall≥X %»</b> = un punto de la tabla de umbrales del modelo: el corte más pequeño que alcanza a capturar ese X % de las ventas. Las ocho son mediciones reales, ninguna es una proyección.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántos leads se contactan en ese escenario. Es <b>la capacidad del equipo</b>: el número de llamadas que se pueden hacer.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>% universo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ese mismo número como porcentaje de los 22 656 leads evaluados. Responde: «¿qué parte de la cartera estoy trabajando?».</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv. modelo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Las ventas que <b>realmente</b> quedaron dentro de esa lista al ordenarla con el modelo. Es un dato observado, no una estimación: por eso es un único número y no tiene margen de error.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Recall</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ese mismo dato como porcentaje de las <b>739 ventas que existían</b> en el universo. Es «cuánto del negocio disponible me llevo».</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — media</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Lo que habría conseguido, en promedio, llamar a la misma cantidad de leads pero elegidos sin criterio. <b>Es el punto de comparación de toda la fila.</b></td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — sd</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La desviación estándar: cuánto varía el azar de un intento a otro. Es la «unidad de suerte» — sirve para medir si una diferencia es grande o no.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — IC 95 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El rango donde cae el azar 95 de cada 100 veces. Todo lo que quede por encima del extremo derecho es <b>mejor que el azar con buena racha</b>.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Δ conversiones</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La resta: ventas del modelo − promedio del azar. <b>Son las ventas adicionales que aporta priorizar</b>, sin hacer ni una llamada más.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lift</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La división: cuántas <b>veces</b> mejor que el azar. 6× = seis veces más ventas con las mismas llamadas. 1× significaría «da igual usar el modelo o no».</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Distancia (σ)</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">A cuántas desviaciones estándar está el modelo por encima del promedio del azar. A partir de 2 σ un resultado ya se considera sólido en estadística; aquí todas las filas pasan de 20 σ, es decir, <b>la diferencia no puede ser casualidad en ningún escenario</b>.</td></tr>
</table>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Qué dice cada fila</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Fila</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Lectura en una frase</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Top 5 % — 1 133 leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Un equipo pequeño o una campaña corta. Captura <b>223 de las 739 ventas (30 %)</b> donde el azar habría dado 37: <b>6,03× más</b>. Es la fila donde priorizar rinde al máximo, pero deja el 70 % del negocio sin tocar.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Top 10 % — 2 266 leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El <b>caso base</b> que usa todo el reporte, por ser una capacidad realista. <b>307 ventas (41,5 %)</b> frente a 74 del azar: <b>+233 ventas, 4,15×</b>. Con una décima parte de la cartera se captura más del 40 % del negocio.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Umbral recall≥50 % — 3 234 leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El primer escenario que se lleva <b>la mitad del negocio</b> (370 ventas). Responde a la pregunta inversa: «para capturar el 50 %, ¿cuántas llamadas necesito?». Respuesta: el 14 % de la cartera. Lift 3,51×.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Umbral recall≥60 % — 4 498 leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Subir del 50 % al 60 % del negocio cuesta <b>1 264 llamadas más</b> y baja el lift a 3,03×. Aquí ya se empieza a ver el precio de ampliar la lista.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Top 20 % — 4 532 leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Prácticamente la misma fila que la anterior (34 leads de diferencia) y da casi el mismo resultado. Funciona como <b>control de coherencia</b>: dos caminos distintos —cortar por porcentaje o cortar por recall objetivo— llegan al mismo punto.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Umbral recall≥70 % — 6 591 leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El <b>punto operativo elegido por el proyecto</b>: el corte más pequeño que captura el 70 % de las ventas (518 de 739). Es también donde la <b>ventaja absoluta es máxima: +303 conversiones</b> sobre el azar. Lift 2,41×.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Top 30 % — 6 797 leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Casi idéntica a la anterior, pero medida por el experimento Monte Carlo. Confirma que el punto operativo cae dentro del rango ya validado con simulación, no en una zona improvisada.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Umbral recall≥80 % — 9 125 leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El escenario de <b>rendimientos decrecientes</b>. Pasar del 70 % al 80 % del negocio exige <b>2 534 llamadas adicionales</b>, el lift cae a 1,99× y la ventaja absoluta <b>ya no crece: +294, menos que en la fila anterior</b>. A partir de aquí, ampliar la lista deja de compensar.</td></tr>
</table>
<p style="margin:16px 0 0; font-weight:700; color:#7B1FA2;">El patrón que recorre toda la tabla</p>
<p style="margin:4px 0 0;">Bajando por las filas, <b>el recall sube</b> (se captura más negocio) pero <b>el lift baja</b> (cada llamada rinde menos), porque se van incorporando leads cada vez peores. La columna <b>Δ conversiones</b> muestra dónde está el equilibrio: crece hasta <b>+303 en el punto operativo (6 591 leads)</b> y a partir de ahí <b>deja de crecer</b>. Ese estancamiento —no una preferencia— es lo que justifica haber cortado ahí y no más abajo.</p>
<p style="margin:12px 0 0; font-size:12.5px; color:#5C4566;">Nota: el modelo aparece con un solo número por fila y el azar con toda una distribución (media, sd, IC) porque <b>el modelo no sortea</b>: aplicado a los mismos datos produce siempre el mismo orden. El azar, en cambio, da un resultado distinto cada vez que se ejecuta.</p>
</div>

---
## 1. Executive Summary

Los números de esta sección se generan directamente desde la tabla maestra. El corte de referencia
es **Top 10 %** —una capacidad de contacto realista— y se contrasta con el **punto operativo elegido
por el proyecto** (recall ≥ 70 %).

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Qué va a ver en el resumen ejecutivo</div>
<p style="margin:0 0 8px;">Cuatro maneras de decir lo mismo, porque cada área de la empresa entiende mejor una de ellas:</p><table style="border-collapse:collapse; width:100%; font-size:13px; margin-top:8px;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Cifra</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Cómo leerla</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conversiones capturadas</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas ventas caen dentro de la lista contactada. El número que le importa a dirección.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>% del negocio disponible</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ese mismo dato como porcentaje de las 739 ventas que existían. Responde: «¿cuánto del pastel me llevo?».</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conversiones por cada 100 contactos</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La productividad de una jornada de llamadas. Es lo que un asesor siente en el día a día.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Contactos necesarios por conversión</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Lo mismo al revés: cuántas llamadas cuesta cerrar una venta. Menos es mejor.</td></tr>
</table><p style="margin:12px 0 0;">Y dos referencias que se repiten: el <b>Top 10 %</b> (una capacidad de contacto realista, usada como caso base) y el <b>punto operativo</b>, que es el nivel que el proyecto eligió para trabajar.</p>
</div>

In [ ]:
def fila(leads):
    return res.loc[(res.leads - leads).abs().idxmin()]

r5, r10, r20, r30 = fila(1133), fila(2266), fila(4532), fila(6797)
rop = fila(6591)     # punto operativo del proyecto (recall >= 70 %)

display(Markdown(f"""
### Los hallazgos en una tabla

| Con una capacidad de {int(r10.leads):,} leads (Top 10 %) | Modelo | Azar (media) | Diferencia |
|---|---:|---:|---:|
| Conversiones capturadas | **{int(r10.conv_modelo)}** | {r10.azar_media:.0f} | **+{r10.delta_conv:.0f}** (+{r10.incremento_pct:.0f} %) |
| % del negocio disponible | **{r10.recall_modelo*100:.1f} %** | {r10.azar_media/N_CONV*100:.1f} % | ×{r10.lift:.2f} |
| Conversiones por cada 100 contactos | **{r10.conv_x100_modelo:.1f}** | {r10.conv_x100_azar:.1f} | ×{r10.lift:.2f} |
| Contactos necesarios por conversión | **{r10.contactos_x_conv_modelo:.1f}** | {r10.contactos_x_conv_azar:.1f} | −{(1-r10.contactos_x_conv_modelo/r10.contactos_x_conv_azar)*100:.0f} % |

### Los hallazgos en palabras

- Sobre un universo de **{N_UNIVERSO:,} leads** que contenía **{N_CONV} conversiones reales**
  (tasa base **{TASA_BASE*100:.2f} %**), el modelo prioriza **{int(r10.leads):,} leads** y dentro de
  ellos **captura {int(r10.conv_modelo)} conversiones**: el **{r10.recall_modelo*100:.1f} %** de todo
  el negocio disponible, contactando solo al 10 % de la cartera.
- Una **selección aleatoria del mismo tamaño** captura en promedio **{r10.azar_media:.1f}**. La
  diferencia es de **{r10.delta_conv:.0f} conversiones adicionales**, un incremento del
  **{r10.incremento_pct:.0f} %** y un **lift de {r10.lift:.2f}×**.
- **Probabilidad de que el azar iguale o supere al modelo: {r10.p_azar_ge_modelo:.1e}.** En las
  {N_MC_HD:,} simulaciones realizadas, el mejor resultado aleatorio fue de **{int(r10.azar_max)}
  conversiones** frente a **{int(r10.conv_modelo)}** del modelo: ni una sola vez el azar se acercó.
- En el **punto operativo del proyecto** ({int(rop.leads):,} leads, {rop.pct_universo:.0f} % del
  universo) el modelo captura **{int(rop.conv_modelo)} conversiones ({rop.recall_modelo*100:.0f} %
  del total)** frente a **{rop.azar_media:.0f}** del azar: **+{rop.delta_conv:.0f} conversiones**,
  lift {rop.lift:.2f}×.
- La ventaja es **mayor cuanto más escasa es la capacidad**: con el 5 % del universo
  ({int(r5.leads):,} leads) el lift sube a **{r5.lift:.2f}×**. El modelo vale más precisamente
  cuando el recurso comercial es más escaso.
"""))

nota(f'<b>La conclusión en una frase.</b> Con la misma capacidad de contactar '
     f'<b>{int(r10.leads):,} leads</b>, el modelo identifica <b>{int(r10.conv_modelo)} conversiones</b>, '
     f'mientras que una selección aleatoria identifica en promedio <b>{r10.azar_media:.0f}</b>. '
     f'Eso son <b>{r10.delta_conv:.0f} conversiones adicionales sin aumentar la carga de trabajo '
     f'comercial</b>: la ventaja no viene de contactar más, sino de contactar mejor.', 'info')

---
## 2. Model vs. Random Selection

Comparación directa entre el **resultado observado** del modelo —un único número por corte, porque
el modelo no es aleatorio: produce un orden fijo— y la **distribución completa** de lo que habría
ocurrido eligiendo leads al azar.

Aquí se calcula además la métrica que el experimento original no guardaba:

$$P(\text{azar} \geq \text{modelo}) = \Pr\big[X \geq m\big], \qquad X \sim \text{Hipergeométrica}(N, K, n)$$

Es decir: **de todas las formas posibles de elegir al azar `n` leads, ¿qué fracción captura al menos
tantas conversiones como el modelo?**

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Qué significa P(azar ≥ modelo)</div>
<p style="margin:0 0 10px;">Esa fórmula responde una pregunta muy concreta: <b>«si en vez de usar el modelo el equipo eligiera leads a ciegas, ¿qué posibilidad tendría de salir tan bien parado como el modelo, solo por suerte?»</b></p><p style="margin:0 0 8px;">Cuanto más pequeño sea ese número, menos creíble es que la ventaja del modelo sea casualidad. Para leerlo:</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;">Un valor como <b>1 × 10⁻³⁰</b> significa un 1 precedido de 30 ceros: no es «poco probable», es <b>imposible en la práctica</b>. Ni repitiendo el sorteo todos los días durante millones de años saldría.</li>
<li style="margin-bottom:6px;">La <b>σ (sigma)</b> mide la distancia del modelo respecto al promedio del azar, usando como regla la variación natural del propio azar. En ciencia se considera «resultado sólido» a partir de <b>2 σ</b>; aquí las distancias son de decenas de σ.</li>
<li style="margin-bottom:6px;">Diferencia clave: esto demuestra que la ventaja <b>es real, no suerte</b>. No dice, por sí solo, si la ventaja es <b>grande en dinero</b> — eso lo responde la Sección 4.</li>
</ul>
</div>

In [ ]:
comp = res[res.origen == 'Monte Carlo §27'].copy()

tabla(pd.DataFrame({
        'Corte'              : [f'Top {p:.0f}%' for p in comp.pct_universo],
        'Leads contactados'  : comp.leads.map('{:,}'.format),
        'MODELO'             : comp.conv_modelo.astype(int),
        'Azar — media'       : comp.azar_media.map('{:.1f}'.format),
        'Azar — mediana'     : comp.azar_mediana.map('{:.0f}'.format),
        'Azar — desv. est.'  : comp.azar_sd.map('{:.2f}'.format),
        'Azar — IC 95 %'     : [f'[{lo:.0f}, {hi:.0f}]' for lo, hi in zip(comp.azar_p2_5, comp.azar_p97_5)],
        'Azar — mínimo'      : comp.azar_min.astype(int),
        'Azar — máximo'      : comp.azar_max.astype(int),
      }),
      titulo='Modelo observado frente a la distribución del azar',
      destacar=['MODELO'],
      nota='Mínimo y máximo corresponden a las 200 000 simulaciones; el IC 95 % es empírico.')

pcts = [1, 5, 25, 50, 75, 95, 99, 99.9]
tp = pd.DataFrame({'Corte': [f'Top {p:.0f}%' for p in comp.pct_universo]})
for p in pcts:
    tp[f'p{p}'] = [f'{np.percentile(dist[int(t)], p):.0f}' for t in comp.leads]
tp['MODELO'] = comp.conv_modelo.astype(int).values
tabla(tp, titulo='Percentiles de la distribución aleatoria — 200 000 iteraciones',
      destacar=['MODELO'],
      nota='Ni el percentil 99,9 del azar se aproxima al resultado del modelo en ningún corte.')

tabla(pd.DataFrame({
        'Corte'                    : [f'Top {p:.0f}%' for p in comp.pct_universo],
        'Modelo'                   : comp.conv_modelo.astype(int),
        'Mejor azar en 200k sim.'  : comp.azar_max.astype(int),
        'Brecha vs. el mejor azar' : (comp.conv_modelo - comp.azar_max).astype(int),
        'Distancia (σ)'            : comp.z_modelo.map('{:.1f}'.format),
        'P(azar ≥ modelo) empírica': [f'0 / {N_MC_HD:,}'] * len(comp),
        'P(azar ≥ modelo) exacta'  : [f'{v:.2e}' for v in comp.p_azar_ge_modelo],
      }),
      titulo='¿Puede el azar igualar al modelo?',
      destacar=['P(azar ≥ modelo) exacta'])

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer las tres tablas de esta sección</div>
<p style="margin:0px 0 0;">Las tres responden la misma pregunta con lupas cada vez más finas: <b>¿podría el azar llegar a donde llegó el modelo?</b></p>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 1 — Modelo observado frente a la distribución del azar</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>MODELO</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Un solo número por fila: las ventas que capturó. No tiene rango porque el modelo no sortea, siempre ordena igual.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — media / mediana</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El resultado típico del azar. Que ambas casi coincidan indica que la distribución es simétrica, sin sorpresas.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — desv. est.</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuánto varía el azar entre un intento y otro.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — IC 95 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El terreno habitual del azar: ahí cae 95 de cada 100 intentos.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar — mínimo / máximo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El peor y el <b>mejor</b> resultado que apareció en 200 000 sorteos. El máximo es el <b>techo de la suerte</b>: ni con la mejor racha de 200 000 intentos el azar pasó de ahí.</td></tr>
</table>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 2 — Percentiles de la distribución aleatoria</p>
<p style="margin:4px 0 0;">Un percentil responde «¿por debajo de qué valor queda el X % de los sorteos?». Leído de izquierda a derecha, es <b>el azar de peor a mejor</b>: p1 = azar en un día pésimo, p50 = azar normal, p99,9 = azar con una racha de una en mil. La última columna repite el resultado del modelo para comparar. <b>Qué mirar:</b> ni la columna p99,9 se acerca a la columna MODELO en ninguna fila.</p>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 3 — ¿Puede el azar igualar al modelo?</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Mejor azar en 200k sim.</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El mejor resultado aleatorio de las 200 000 simulaciones.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Brecha vs. el mejor azar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas ventas le saca el modelo <b>a ese mejor caso</b>, no al promedio. Es la comparación más exigente posible.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Distancia (σ)</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">A cuántas desviaciones estándar está el modelo del promedio del azar. En estadística, 2 σ ya se considera un resultado sólido.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>P(azar ≥ modelo) empírica</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas de las 200 000 simulaciones igualaron al modelo. Dice <b>0 / 200 000</b> en los cuatro cortes: ninguna.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>P(azar ≥ modelo) exacta</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La misma probabilidad calculada con la fórmula, que sí puede medir lo que la simulación no alcanza a ver. Valores como <b>3e-117</b> significan un 3 con 117 ceros por delante: imposible en la práctica.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Qué mirar:</b> la simulación dice «nunca pasó» y la fórmula dice «no puede pasar». Las dos vías, independientes entre sí, llegan a la misma conclusión.</p>
</div>

In [ ]:
p10 = r10.p_azar_ge_modelo
display(Markdown(f"""
### Cómo leer P(azar ≥ modelo)

En el corte Top 10 % el valor exacto es **{p10:.1e}**. Traducido:

- Si el equipo comercial eligiera **{int(r10.leads):,} leads al azar todos los días**, tendría que
  repetir el ejercicio del orden de **10^{abs(np.log10(p10)):.0f} veces** para obtener por pura
  suerte un resultado tan bueno como el del modelo. Es un número sin significado físico: **el azar
  no puede llegar ahí**.
- El resultado del modelo está a **{r10.z_modelo:.1f} desviaciones estándar** por encima de la media
  aleatoria. Como referencia, el umbral convencional de significancia (p < 0,05) está a ~2 σ.
- En las **{N_MC_HD:,} simulaciones** ejecutadas, el mejor resultado aleatorio fue **{int(r10.azar_max)}
  conversiones**; el modelo obtuvo **{int(r10.conv_modelo)}**. La brecha con el *mejor caso* del azar
  es de **{int(r10.conv_modelo - r10.azar_max)} conversiones**.
"""))

nota('<b>Qué significa y qué no.</b> Este valor demuestra que la ventaja <b>no es producto de la '
     'suerte en la partición de datos ni de una casualidad del conjunto de evaluación</b>. No dice '
     'nada, por sí solo, sobre el <i>tamaño comercial</i> de la ventaja: para eso están las secciones '
     '4 y 5. Un resultado puede ser estadísticamente incontestable y comercialmente irrelevante — '
     'aquí se verifican las dos cosas por separado.', 'alerta')

---
## 3. Monte Carlo Distribution Analysis

El promedio del azar es solo un resumen. Lo que decide si la ventaja del modelo es sólida es la
**forma completa** de la distribución aleatoria: cuánto varía, hasta dónde llegan sus mejores casos
y dónde queda el modelo dentro de ese rango.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo se lee la «forma» de la distribución del azar</div>
<p style="margin:0 0 8px;">Al sortear 200 000 veces no sale un único número, sale un abanico de resultados posibles. Estos son los términos que describen ese abanico:</p><table style="border-collapse:collapse; width:100%; font-size:13px; margin-top:8px;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Término</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Traducción</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Media</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El promedio de todos los sorteos: lo que el azar da «en un día normal».</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Mediana</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El valor del medio: la mitad de los sorteos queda por debajo y la mitad por encima. Si media y mediana casi coinciden, el abanico es simétrico.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Desviación estándar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuánto se aleja del promedio un sorteo típico. Es la «unidad de suerte» del problema.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Coeficiente de variación</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Esa misma dispersión en porcentaje del promedio, para poder comparar cortes de distinto tamaño. Alto = azar más errático.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Asimetría</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Si el abanico está inclinado hacia un lado. Cerca de 0 = campana pareja.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Percentil 95 (p95)</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El valor que solo supera el 5 % de los sorteos: el azar «con buena racha». El p99,9 es el azar con una racha extraordinaria.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>IC 95 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El rango donde caen 95 de cada 100 sorteos. Es el terreno normal del azar; todo lo que quede fuera es excepcional.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Rango observado</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El peor y el mejor resultado que apareció en las 200 000 simulaciones. Es el techo de la suerte.</td></tr>
</table><p style="margin:12px 0 0;">La conclusión que se busca es simple: <b>¿el resultado del modelo cabe dentro de ese abanico o queda completamente fuera?</b> Si queda fuera incluso del mejor caso del azar, ninguna racha de suerte explica la diferencia.</p>
</div>

In [ ]:
tabla(pd.DataFrame({
        'Corte'                : [f'Top {p:.0f}%' for p in comp.pct_universo],
        'Media'                : comp.azar_media.map('{:.2f}'.format),
        'Mediana'              : comp.azar_mediana.map('{:.0f}'.format),
        'Desv. estándar'       : comp.azar_sd.map('{:.2f}'.format),
        'Coef. de variación'   : (comp.azar_sd / comp.azar_media * 100).map('{:.1f} %'.format),
        'Asimetría'            : comp.azar_asimetria.map('{:+.3f}'.format),
        'Rango observado'      : [f'{lo} – {hi}' for lo, hi in zip(comp.azar_min, comp.azar_max)],
        'Amplitud del IC 95 %' : (comp.azar_p97_5 - comp.azar_p2_5).astype(int),
        'Percentil del modelo' : [f'> {100*(1 - 1/N_MC_HD):.4f}'] * len(comp),
      }),
      titulo='Forma de la distribución aleatoria')

display(Markdown(f"""
**Qué muestran estos números:**

- **La dispersión relativa cae al aumentar la capacidad**: el coeficiente de variación pasa de
  **{comp.iloc[0].azar_sd/comp.iloc[0].azar_media*100:.1f} %** en Top 5 % a
  **{comp.iloc[-1].azar_sd/comp.iloc[-1].azar_media*100:.1f} %** en Top 30 %. Con pocos contactos el
  azar es **más errático**: la lotería comercial es más extrema cuando se llama a poca gente.
- **La distribución es prácticamente simétrica** (asimetría entre
  {comp.azar_asimetria.min():+.3f} y {comp.azar_asimetria.max():+.3f}). No hay una cola larga por la
  derecha que permita "golpes de suerte" excepcionales: el azar no tiene un escenario optimista
  oculto.
- **En ningún corte el máximo de {N_MC_HD:,} simulaciones alcanza al modelo.** El modelo no está en
  el extremo de la distribución aleatoria; está fuera de ella.
"""))

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer la tabla de forma de la distribución</div>
<p style="margin:0px 0 0;">Esta tabla describe <b>el abanico de resultados del azar</b>, sin mirar todavía al modelo. Cada fila es un corte; cada columna, un rasgo de ese abanico.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Media / Mediana</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El centro del abanico visto de dos formas: el promedio y el valor que parte los sorteos por la mitad. Si coinciden, el abanico es parejo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Desv. estándar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El ancho típico del abanico, en conversiones.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Coef. de variación</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ese ancho como porcentaje del promedio. Sirve para comparar cortes de tamaños distintos, cosa que la desviación en bruto no permite.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Asimetría</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Si el abanico está inclinado. <b>0 = campana simétrica</b>; positivo = cola larga hacia la derecha, es decir, existirían «golpes de suerte» raros pero enormes.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Rango observado</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El peor y el mejor resultado en 200 000 sorteos: el abanico completo, de extremo a extremo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Amplitud del IC 95 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas conversiones de ancho tiene la zona habitual del azar.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Percentil del modelo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">En qué lugar del abanico cae el modelo. Dice «> 99,9995» porque <b>superó a las 200 000 simulaciones sin excepción</b>: no hay percentil más alto que se pueda medir con esa cantidad.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Qué mirar, en dos lecturas:</b></p>
<ul style="margin:6px 0 0; padding-left:20px;"><li style="margin-bottom:6px;">El <b>coeficiente de variación baja al bajar por las filas</b> (de 15,8 % en Top 5 % a 5,5 % en Top 30 %): con pocas llamadas el azar es más errático — la lotería comercial es más extrema cuanto menos se llama.</li><li style="margin-bottom:6px;">La <b>asimetría es casi cero</b> en todas las filas: el azar <b>no tiene un escenario optimista escondido</b>. No existe la racha milagrosa que lo acerque al modelo.</li></ul>
</div>

### Gráfico principal — dónde cae el modelo dentro de la distribución del azar

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer estos cuatro histogramas</div>
<p style="margin:0 0 8px;">Cada recuadro es un nivel de capacidad distinto. Dentro de cada uno:</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;">La <b style="color:#eb6834;">montaña naranja</b> son las 200 000 selecciones al azar: su altura indica cuántas veces salió cada resultado. Es «todo lo que el azar puede dar».</li>
<li style="margin-bottom:6px;">La <b style="color:#eb6834;">franja sombreada</b> es el intervalo del 95 %: el terreno habitual del azar.</li>
<li style="margin-bottom:6px;">La <b style="color:#eb6834;">línea naranja punteada</b> (más oscura) es el promedio del azar.</li>
<li style="margin-bottom:6px;">La <b style="color:#2a78d6;">línea azul</b> es el resultado del modelo. Es una sola línea, no una montaña, porque el modelo no sortea: da siempre el mismo orden.</li>
<li style="margin-bottom:6px;">La <b>flecha con la etiqueta</b> mide la distancia entre ambos, en conversiones y en σ.</li>
</ul><p style="margin:10px 0 0;">Lo único que hay que mirar: <b>la línea azul no toca la montaña naranja en ningún corte</b>. No están cerca, no se rozan.</p>
</div>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.6))
for ax, (_, r) in zip(axes.flat, comp.iterrows()):
    tn = int(r.leads); d = dist[tn]
    paso = max(1, (d.max() - d.min()) // 45)
    ax.hist(d, bins=np.arange(d.min() - 0.5, d.max() + 1.5, paso), color=C_AZAR, alpha=0.80,
            edgecolor='none', label=f'Selección aleatoria\n({N_MC_HD:,} simulaciones)')
    ax.axvspan(r.azar_p2_5, r.azar_p97_5, color=C_AZAR, alpha=0.18, lw=0)
    ax.axvline(r.azar_media, color='#8a3a12', lw=2, ls='--', label=f'Media del azar = {r.azar_media:.0f}')
    ax.axvline(r.conv_modelo, color=C_MODELO, lw=2.6, label=f'MODELO = {int(r.conv_modelo)}')
    ax.set_xlim(0, r.conv_modelo * 1.12)
    ymax = ax.get_ylim()[1]
    ax.annotate('', xy=(r.conv_modelo, ymax * 0.60), xytext=(r.azar_media, ymax * 0.60),
                arrowprops=dict(arrowstyle='<->', color=INK2, lw=1.1))
    ax.text((r.azar_media + r.conv_modelo) / 2, ymax * 0.635, f'+{r.delta_conv:.0f} conv.\n({r.z_modelo:.0f} σ)',
            ha='center', va='bottom', fontsize=9, color=INK, fontweight='bold')
    ax.set_title(f'Top {r.pct_universo:.0f}% — {tn:,} leads contactados', fontweight='bold', fontsize=11)
    ax.set_xlabel('Conversiones capturadas'); ax.set_ylabel('Frecuencia (simulaciones)')
    ax.set_yticklabels([])
    ax.legend(fontsize=8, loc='upper center')
fig.suptitle('Distribución Monte Carlo de la selección aleatoria vs. resultado del modelo\n'
             f'Universo H — {N_UNIVERSO:,} leads, {N_CONV} conversiones (tasa base {TASA_BASE*100:.2f} %)',
             fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout(); plt.savefig(FIG_DIR / 'mc_distribucion_vs_modelo.png', bbox_inches='tight'); plt.show()

**Qué muestra.** En los cuatro cortes, la nube naranja —todo lo que el azar *puede* producir— y la
línea azul del modelo **no se solapan en absoluto**. La banda sombreada es el intervalo del 95 % de
la selección aleatoria; el modelo no está en su borde, está muy lejos de su extremo superior.

**Por qué importa.** Si las distribuciones se solaparan, la elección entre modelo y azar sería una
apuesta: algunos meses el azar ganaría. Aquí la separación es tan amplia que **el peor resultado
plausible del modelo sigue estando por encima del mejor resultado plausible del azar** en este
conjunto de evaluación.

**Qué significa comercialmente.** La priorización no es un ajuste marginal sobre la práctica actual.
Cambia el orden de magnitud del rendimiento de una misma jornada de trabajo: la campana naranja *es*
el desempeño esperado de un equipo que trabaja su cartera sin criterio de priorización.

---
## 4. Incremental Value of the Model

La pregunta ya no es *si* el modelo es mejor, sino **cuánto vale la diferencia** en unidades que el
negocio usa: conversiones, contactos y esfuerzo.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Las dos maneras de cobrar la ventaja</div>
<p style="margin:0 0 8px;">La diferencia entre el modelo y el azar se puede aprovechar de dos formas opuestas, y esta sección mide las dos:</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;"><b>Mismo esfuerzo → más ventas.</b> Se hacen las mismas llamadas de siempre, pero en otro orden, y se cierran más conversiones. La ganancia sale de <i>a quién</i> se llama, no de llamar más.</li>
<li style="margin-bottom:6px;"><b>Mismo resultado → menos esfuerzo.</b> Si la meta es igualar lo que hoy se consigue, basta con muchas menos llamadas. Las llamadas que sobran quedan libres para otra cosa.</li>
</ul><p style="margin:12px 0 0;"><b>«Contactos improductivos»</b> son las llamadas que no terminaron en venta. Siempre son la mayoría —la tasa base es del 3 %—; lo relevante es que con el modelo son bastantes menos por cada venta lograda.</p><p style="margin:8px 0 0;"><b>Lift 3×</b> quiere decir «tres veces lo del azar». <b>Lift 1×</b> querría decir «igual que no hacer nada».</p>
</div>

In [ ]:
tabla(pd.DataFrame({
        'Corte'              : [f'Top {p:.0f}%' for p in comp.pct_universo],
        'Leads'              : comp.leads.map('{:,}'.format),
        'Conv. modelo'       : comp.conv_modelo.astype(int),
        'Conv. azar'         : comp.azar_media.map('{:.1f}'.format),
        'Conv. adicionales'  : comp.delta_conv.map('+{:.0f}'.format),
        'Incremento'         : comp.incremento_pct.map('+{:.0f} %'.format),
        'Lift'               : comp.lift.map('{:.2f}×'.format),
      }),
      titulo='A. Ventaja en conversiones', destacar=['Conv. adicionales', 'Lift'])

tabla(pd.DataFrame({
        'Corte'                        : [f'Top {p:.0f}%' for p in comp.pct_universo],
        'Conv./100 contactos — modelo' : comp.conv_x100_modelo.map('{:.1f}'.format),
        'Conv./100 contactos — azar'   : comp.conv_x100_azar.map('{:.1f}'.format),
        'Contactos/conversión — modelo': comp.contactos_x_conv_modelo.map('{:.1f}'.format),
        'Contactos/conversión — azar'  : comp.contactos_x_conv_azar.map('{:.1f}'.format),
        'Contactos improd. — modelo'   : comp.improductivos_modelo.astype(int).map('{:,}'.format),
        'Contactos improd. — azar'     : comp.improductivos_azar.round(0).astype(int).map('{:,}'.format),
      }),
      titulo='B. Eficiencia operativa',
      nota='«Contactos improductivos» = contactos realizados que no terminan en conversión.')

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer las tablas A y B</div>
<p style="margin:0px 0 0;">Las dos miden la misma ventaja, pero en unidades distintas: <b>A</b> en ventas y <b>B</b> en esfuerzo de llamadas.</p>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla A — Ventaja en conversiones</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv. modelo / Conv. azar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Lo que captura cada estrategia con el mismo número de llamadas.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv. adicionales</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La resta. <b>Son ventas extra sin hacer ni una llamada más</b>: salen de cambiar el orden, no de trabajar más.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Incremento</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Esa misma diferencia en porcentaje sobre lo que da el azar.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lift</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Y esa misma diferencia como multiplicador. <b>Incremento y lift dicen lo mismo</b>: +315 % es exactamente lo mismo que 4,15×. Se muestran los dos porque cada área de la empresa está acostumbrada a uno.</td></tr>
</table>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla B — Eficiencia operativa</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv./100 contactos</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">De cada 100 llamadas, cuántas terminan en venta. Es la <b>productividad de una jornada</b>. (Es la precisión, expresada en la unidad en que la vive el equipo.)</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Contactos/conversión</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Lo mismo al revés: cuántas llamadas cuesta cerrar una venta. <b>Aquí menos es mejor.</b></td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Contactos improd.</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Llamadas que no terminaron en venta. Siempre son la mayoría —la tasa base es del 3 %—; lo relevante es cuántas se gastan <b>por cada venta lograda</b>, no su total.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Qué mirar:</b> en la tabla A la ventaja se ve como «más resultado»; en la B, como «menos trabajo por resultado». Son la misma cifra contada desde los dos lados del mostrador.</p>
</div>

In [ ]:
# ── Reducción de esfuerzo: ¿cuántos contactos necesita el modelo para igualar al azar? ──
gx = np.concatenate([[0], res.leads.values])          # curva de ganancia del modelo
gy = np.concatenate([[0], res.conv_modelo.values])

esf = []
for _, r in comp.iterrows():
    objetivo  = r.azar_media                          # lo que el azar logra con r.leads contactos
    leads_mod = np.interp(objetivo, gy, gx)           # contactos que el modelo necesita para eso
    esf.append({'Escenario': f'Igualar al azar en Top {r.pct_universo:.0f}%',
                'Contactos del azar': int(r.leads),
                'Conversiones a igualar': round(objetivo, 1),
                'Contactos que necesita el modelo': int(round(leads_mod)),
                'Contactos ahorrados': int(round(r.leads - leads_mod)),
                'Reducción de esfuerzo': f'−{(1 - leads_mod / r.leads) * 100:.0f} %'})
esf = pd.DataFrame(esf)

tabla(pd.DataFrame({
        'Escenario'                  : esf.Escenario,
        'Contactos del azar'         : esf['Contactos del azar'].map('{:,}'.format),
        'Conversiones a igualar'     : esf['Conversiones a igualar'].map('{:.1f}'.format),
        'Contactos del modelo'       : esf['Contactos que necesita el modelo'].map('{:,}'.format),
        'Contactos ahorrados'        : esf['Contactos ahorrados'].map('{:,}'.format),
        'Reducción de esfuerzo'      : esf['Reducción de esfuerzo'],
      }),
      titulo='C. Reducción de esfuerzo comercial, a igualdad de resultado',
      destacar=['Reducción de esfuerzo'],
      nota='Interpolación lineal sobre los 8 puntos reales de la curva de ganancia. '
           '<b>Los cuatro objetivos caen dentro del primer tramo medido (0–1 133 leads)</b>, donde la '
           'interpolación es lineal desde el origen; por eso el porcentaje ahorrado es idéntico en los '
           'cuatro casos e igual al lift del Top 5 %. Es una estimación <b>conservadora</b>: la curva '
           'real es más empinada en la cabeza del ranking, así que el modelo necesitaría aún menos contactos.')

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer la tabla C</div>
<p style="margin:0px 0 0;">Esta tabla invierte la pregunta. En vez de «con las mismas llamadas, ¿cuánto más vendo?», pregunta: <b>«para vender lo mismo que hoy, ¿cuántas llamadas me ahorro?»</b>. Es la lectura que interesa cuando la meta comercial ya está fijada.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Escenario</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La meta a igualar: lo que consigue el azar en ese corte.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Contactos del azar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Las llamadas que hace hoy una operación sin priorizar.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conversiones a igualar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Las ventas que consigue con esas llamadas. <b>Es el objetivo.</b></td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Contactos del modelo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Las llamadas que necesita el modelo para llegar a esa misma cifra, leídas sobre su curva de ganancia.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Contactos ahorrados</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La diferencia: llamadas que quedan libres para otra cosa.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Reducción de esfuerzo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ese ahorro en porcentaje.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Por qué el porcentaje sale idéntico en las cuatro filas:</b> no es un error. Las cuatro metas son tan modestas para el modelo que las cumple dentro de su <b>primer tramo</b> (los primeros 1 133 leads), y dentro de un mismo tramo la interpolación es una línea recta, así que la proporción ahorrada es siempre la misma.</p>
<p style="margin:10px 0 0;"><b>Es además una estimación conservadora:</b> la curva real es más empinada en la cabeza del ranking que la línea recta usada para interpolar, así que en la práctica el modelo necesitaría <b>todavía menos</b> llamadas que las que dice la tabla.</p>
</div>

In [ ]:
e10 = esf.iloc[1]
display(Markdown(f"""
### Interpretación del valor incremental

**1. A igual esfuerzo, más resultado.** Con la misma capacidad de contactar **{int(r10.leads):,} leads**,
el modelo identifica **{int(r10.conv_modelo)} conversiones** frente a **{r10.azar_media:.0f}** de una
selección aleatoria: **{r10.delta_conv:.0f} conversiones adicionales (+{r10.incremento_pct:.0f} %)**
obtenidas **sin llamar a un solo lead más**. La ganancia es puramente de asignación, no de volumen.

**2. A igual resultado, menos esfuerzo.** La lectura inversa es igual de importante: para capturar
las **{e10['Conversiones a igualar']:.0f} conversiones** que el azar consigue contactando
**{e10['Contactos del azar']:,} leads**, al modelo le bastan **{e10['Contactos que necesita el modelo']:,}
contactos**. Es una reducción de esfuerzo del **{e10['Reducción de esfuerzo'].strip('− ')}**, o
**{e10['Contactos ahorrados']:,} llamadas liberadas** para otras actividades.

**3. Productividad por contacto.** El modelo eleva la tasa de éxito de **{r10.conv_x100_azar:.1f} a
{r10.conv_x100_modelo:.1f} conversiones por cada 100 contactos**. En términos operativos, un asesor
pasa de necesitar **{r10.contactos_x_conv_azar:.0f} contactos por venta** a necesitar
**{r10.contactos_x_conv_modelo:.0f}**.

**4. El lift crece cuando la capacidad cae.** De Top 30 % a Top 5 % el lift pasa de **{r30.lift:.2f}×
a {r5.lift:.2f}×**. La priorización rinde más cuanto más restringido está el recurso, que es
exactamente el escenario en el que una herramienta de priorización se justifica.
"""))

nota(f'<b>El matiz honesto sobre los contactos improductivos.</b> De los {int(r10.leads):,} contactos, '
     f'el azar desperdicia en promedio <b>{r10.improductivos_azar:.0f}</b> y el modelo '
     f'<b>{int(r10.improductivos_modelo):,}</b>. La reducción es modesta en términos absolutos, y '
     f'<b>lo es realmente</b>: con una tasa base de {TASA_BASE*100:.2f} %, la inmensa mayoría de los '
     f'contactos serán improductivos con cualquier estrategia. El modelo <b>no elimina el trabajo en '
     f'vano; concentra el trabajo útil</b>. Conviene comunicar esta distinción con claridad al equipo '
     f'comercial.', 'alerta')

### Gráfico — el modelo frente al azar y su banda de incertidumbre, a lo largo de la capacidad

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer la curva de capacidad</div>
<p style="margin:0 0 8px;">El eje horizontal es <b>cuántos leads se contactan</b> y el vertical <b>cuántas ventas se capturan</b>. Hay dos líneas:</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;">La <b style="color:#2a78d6;">azul</b> es el modelo; la <b style="color:#eb6834;">naranja</b>, el azar. La <b>distancia vertical entre ellas</b>, a cualquier altura, es lo que aporta priorizar con ese presupuesto de llamadas.</li>
<li style="margin-bottom:6px;">La banda naranja clara es el margen de incertidumbre del azar. Es tan estrecha que casi no se ve: el azar es predecible en su mediocridad.</li>
<li style="margin-bottom:6px;">La línea punteada horizontal marca el <b>techo</b>: las 739 conversiones que existen. Nadie puede pasar de ahí.</li>
</ul><p style="margin:10px 0 0;">La curva azul se va aplanando: <b>los primeros leads de la lista valen mucho más que los últimos</b>. Eso se llama <b>rendimientos decrecientes</b> y es lo que justifica elegir un punto de corte en lugar de llamar a todo el mundo.</p>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(res.leads, res.azar_p2_5, res.azar_p97_5, color=C_AZAR, alpha=0.20, lw=0,
                label='Azar — intervalo 95 %')
ax.plot(res.leads, res.azar_media, 's--', color=C_AZAR, lw=2, ms=7, label='Azar — media esperada')
ax.plot(res.leads, res.conv_modelo, 'o-', color=C_MODELO, lw=2.4, ms=8, label='Modelo (observado)')
ax.axhline(N_CONV, color=INK2, lw=1, ls=':')
ax.text(res.leads.max(), N_CONV + 8, f'Total de conversiones existentes = {N_CONV}',
        ha='right', fontsize=9, color=INK2)

for _, r in comp.iterrows():   # etiquetas directas solo en los 4 cortes del experimento
    ax.annotate(f'{int(r.conv_modelo)}', (r.leads, r.conv_modelo), textcoords='offset points',
                xytext=(0, 11), ha='center', fontsize=9, fontweight='bold', color=INK)
    ax.annotate(f'{r.azar_media:.0f}', (r.leads, r.azar_media), textcoords='offset points',
                xytext=(0, -16), ha='center', fontsize=9, color=INK2)

ax.set_xlabel('Leads contactados (capacidad comercial)', fontweight='bold')
ax.set_ylabel('Conversiones capturadas', fontweight='bold')
ax.set_title('Conversiones capturadas según la capacidad de contacto\n'
             'Modelo observado vs. selección aleatoria con su incertidumbre',
             fontweight='bold', fontsize=12)
ax.legend(loc='lower right'); ax.set_xlim(0, res.leads.max() * 1.03); ax.set_ylim(0, N_CONV * 1.12)
plt.tight_layout(); plt.savefig(FIG_DIR / 'mc_modelo_vs_azar_capacidad.png', bbox_inches='tight'); plt.show()

**Qué muestra.** La curva azul se despega de la naranja desde el primer tramo y la banda de
incertidumbre del azar —tan estrecha que apenas se distingue de su línea central— nunca la alcanza.
La distancia vertical entre ambas curvas *es* el valor incremental del modelo en cada nivel de
capacidad.

**Por qué importa.** La curva azul es cóncava: **los primeros leads priorizados aportan mucho más
que los siguientes**. Entre 0 y 1 133 contactos el modelo suma 223 conversiones; entre 4 532 y 6 797
(2 265 contactos más, el doble) suma apenas 73. Existe un punto de rendimientos decrecientes, y
**ampliar la capacidad de contacto no compensa perder la priorización**.

**Qué significa comercialmente.** El equipo puede elegir su punto en esta curva según su capacidad
real. La curva naranja marca el costo de oportunidad: es literalmente lo que la operación deja sobre
la mesa cada ciclo si trabaja la cartera sin priorizar.

---
## 5. Commercial Interpretation

Esta sección traduce los resultados a las decisiones que toma un equipo comercial. No se trata de
renombrar métricas, sino de responder qué cambia en el trabajo diario y si el cambio es
suficientemente grande para justificar una modificación del proceso.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">De métricas a decisiones</div>
<p style="margin:0;">Esta sección no introduce cálculos nuevos: toma los mismos resultados y responde qué cambia el lunes por la mañana. Las preguntas concretas son: <b>¿cuántas llamadas hace un asesor por cada venta?</b>, <b>¿cuánto negocio se queda sin tocar un equipo que no prioriza?</b> y <b>¿la mejora es lo bastante grande como para cambiar el proceso de trabajo?</b> Los ejemplos usan una jornada realista (30 contactos al día, 150 a la semana) para que la diferencia se pueda comparar con la experiencia del propio equipo.</p>
</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))
x, w = np.arange(len(comp)), 0.38
lbl = [f'Top {p:.0f}%\n({n:,} leads)' for p, n in zip(comp.pct_universo, comp.leads)]

ax = axes[0]
ax.bar(x - w/2, comp.conv_x100_modelo, w, color=C_MODELO, label='Modelo')
ax.bar(x + w/2, comp.conv_x100_azar,  w, color=C_AZAR,   label='Selección aleatoria')
for i, (m, a) in enumerate(zip(comp.conv_x100_modelo, comp.conv_x100_azar)):
    ax.text(i - w/2, m + 0.25, f'{m:.1f}', ha='center', fontsize=9, fontweight='bold', color=INK)
    ax.text(i + w/2, a + 0.25, f'{a:.1f}', ha='center', fontsize=9, color=INK2)
ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=9)
ax.set_ylabel('Conversiones por cada 100 contactos', fontweight='bold')
ax.set_title('Productividad de cada contacto', fontweight='bold')
ax.set_ylim(0, 23); ax.legend(loc='upper right')

ax = axes[1]
ax.bar(x - w/2, comp.contactos_x_conv_modelo, w, color=C_MODELO, label='Modelo')
ax.bar(x + w/2, comp.contactos_x_conv_azar,  w, color=C_AZAR,   label='Selección aleatoria')
for i, (m, a) in enumerate(zip(comp.contactos_x_conv_modelo, comp.contactos_x_conv_azar)):
    ax.text(i - w/2, m + 0.8, f'{m:.0f}', ha='center', fontsize=9, fontweight='bold', color=INK)
    ax.text(i + w/2, a + 0.8, f'{a:.0f}', ha='center', fontsize=9, color=INK2)
ax.set_xticks(x); ax.set_xticklabels(lbl, fontsize=9)
ax.set_ylabel('Contactos necesarios por conversión', fontweight='bold')
ax.set_title('Esfuerzo requerido por cada venta', fontweight='bold')
ax.set_ylim(0, 36); ax.legend(loc='upper left')

fig.suptitle('Eficiencia comercial: mismo esfuerzo, distinto rendimiento',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(FIG_DIR / 'mc_eficiencia_comercial.png', bbox_inches='tight'); plt.show()

In [ ]:
display(Markdown(f"""
### ¿Qué gana un asesor que usa la priorización?

Un asesor no gana "un mejor modelo": gana **orden de trabajo**. De cada 100 llamadas,
**{r10.conv_x100_modelo:.1f} terminan en conversión** en lugar de **{r10.conv_x100_azar:.1f}**. En la
práctica cotidiana equivale a pasar de una venta cada ~{r10.contactos_x_conv_azar:.0f} contactos a
una cada ~{r10.contactos_x_conv_modelo:.0f}. Si un asesor hace 30 contactos diarios —150 a la
semana—, el azar le entrega en promedio **{150*TASA_BASE:.1f} conversiones semanales**; con la lista
priorizada del Top 10 %, **{150*r10.conv_x100_modelo/100:.1f}**.

### ¿Qué significa para un equipo con capacidad limitada?

El recurso escaso es la capacidad de contacto, no los leads. En este universo hay
**{N_UNIVERSO:,} leads y solo {N_CONV} conversiones posibles**. Un equipo que puede trabajar el 10 %
de la cartera se enfrenta a esta elección:

- **Sin priorización**: captura ~{r10.azar_media:.0f} de las {N_CONV} conversiones
  (**{r10.azar_media/N_CONV*100:.0f} % del negocio disponible**).
- **Con priorización**: captura {int(r10.conv_modelo)} (**{r10.recall_modelo*100:.0f} % del negocio
  disponible**).

El equipo **no trabaja más horas**; recupera {r10.delta_conv:.0f} oportunidades que de otro modo se
habrían quedado en el 90 % de la cartera que nunca llegó a contactarse.

### ¿El modelo concentra las oportunidades reales en un grupo menor de leads?

Sí, y esa es exactamente su función. El **{r5.pct_universo:.0f} % mejor rankeado**
({int(r5.leads):,} leads) contiene **{r5.recall_modelo*100:.0f} % de todas las conversiones**, una
concentración de **{r5.lift:.1f}×** respecto a lo que le correspondería por tamaño. El
**{r10.pct_universo:.0f} %** contiene el **{r10.recall_modelo*100:.0f} %**, y el
**{rop.pct_universo:.0f} %** llega al **{rop.recall_modelo*100:.0f} %**. Es decir: **7 de cada 10
conversiones viven dentro de menos de un tercio de la cartera**, y el modelo sabe cuál es ese tercio.

### ¿Cuál es el costo de oportunidad de trabajar leads al azar?

Con capacidad del 10 %, trabajar sin priorización cuesta **{r10.delta_conv:.0f} conversiones por
ciclo de campaña**. Ese es el número que debe compararse contra el costo de implantar y mantener el
scoring. No es una diferencia estadística abstracta: es negocio que ya estaba en la base de datos y
no se contactó.

### ¿La mejora es suficientemente grande para cambiar una decisión operativa?

Conviene separar los dos juicios:

- *Estadísticamente*: la ventaja es incontestable ({r10.z_modelo:.0f} σ, P(azar ≥ modelo) ≈
  {r10.p_azar_ge_modelo:.0e}). No hay ambigüedad.
- *Comercialmente*: un **{r10.incremento_pct:.0f} % más de conversiones a costo operativo constante**
  es material para cualquier operación de ventas. Un cambio de esta magnitud normalmente exigiría
  contratar más asesores o comprar más leads; aquí se obtiene reordenando una lista.
"""))

nota(f'<b>Lo que el modelo no hace.</b> La precisión absoluta sigue siendo baja: '
     f'<b>{r10.conv_x100_modelo:.1f} %</b> en el Top 10 %. El asesor seguirá recibiendo un «no» en la '
     f'gran mayoría de sus contactos. El modelo <b>no convierte la prospección en una actividad de '
     f'alta tasa de éxito</b>; la vuelve <b>{r10.lift:.1f} veces más eficiente que la alternativa '
     f'realista</b>. Vender la herramienta como algo distinto generaría una expectativa que los datos '
     f'no sostienen, y minaría su adopción.', 'alerta')

### Gráfico — concentración de las oportunidades

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Curva de ganancia y curva de lift</div>
<p style="margin:0 0 8px;">Son dos formas de mirar la <b>concentración</b> de las oportunidades:</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;"><b>Izquierda (curva de ganancia):</b> «contactando al X % de la cartera, capturo el Y % de las ventas». La diagonal naranja es el azar, donde 10 % de llamadas da 10 % de ventas. Cuanto más se despega la curva azul de esa diagonal, mejor ordena el modelo. El área sombreada entre ambas <b>es el negocio que se rescata</b>.</li>
<li style="margin-bottom:6px;"><b>Derecha (curva de lift):</b> cuántas veces mejor que el azar es cada nivel de capacidad. Baja al avanzar porque la cabeza de la lista concentra a los mejores leads; al llegar al 100 % de la cartera el lift vale 1× — ya no hay nada que priorizar, se llamó a todos.</li>
</ul><p style="margin:10px 0 0;">Conclusión práctica: <b>el modelo vale precisamente porque la capacidad es limitada</b>. Cuanto más saturado está el equipo, más aporta.</p>
</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))
pct_leads = np.concatenate([[0], res.pct_universo.values])
pct_conv  = np.concatenate([[0], (res.conv_modelo / N_CONV * 100).values])

ax = axes[0]
ax.plot(pct_leads, pct_conv, 'o-', color=C_MODELO, lw=2.4, ms=7, label='Modelo')
ax.plot([0, 100], [0, 100], '--', color=C_AZAR, lw=2, label='Selección aleatoria')
ax.fill_between(pct_leads, np.interp(pct_leads, [0, 100], [0, 100]), pct_conv,
                color=C_MODELO, alpha=0.13, lw=0)
ax.annotate(f'{r10.pct_universo:.0f} % de los leads →\n{r10.recall_modelo*100:.0f} % de las conversiones',
            xy=(r10.pct_universo, r10.recall_modelo*100), xytext=(19, 26),
            fontsize=9.5, color=INK, arrowprops=dict(arrowstyle='->', color=INK2, lw=1))
ax.set_xlabel('% de leads contactados', fontweight='bold')
ax.set_ylabel('% de conversiones capturadas', fontweight='bold')
ax.set_title('Curva de ganancia acumulada', fontweight='bold')
ax.set_xlim(0, 45); ax.set_ylim(0, 100); ax.legend(loc='lower right')

ax = axes[1]
ax.plot(res.pct_universo, res.lift, 'o-', color=C_MODELO, lw=2.4, ms=7, label='Lift del modelo')
ax.axhline(1, color=C_AZAR, ls='--', lw=2, label='Selección aleatoria (lift = 1)')
for _, r in comp.iterrows():
    ax.annotate(f'{r.lift:.2f}×', (r.pct_universo, r.lift), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold', color=INK)
ax.set_xlabel('% de leads contactados (capacidad)', fontweight='bold')
ax.set_ylabel('Lift (veces mejor que el azar)', fontweight='bold')
ax.set_title('El valor del modelo crece cuando la capacidad es escasa', fontweight='bold')
ax.set_ylim(0, 7); ax.legend(loc='upper right')

fig.suptitle('Concentración de oportunidades: cuánto negocio vive en la cabeza del ranking',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(FIG_DIR / 'mc_concentracion_lift.png', bbox_inches='tight'); plt.show()

**Qué muestra.** El área azul entre las dos curvas de la izquierda es el negocio que el modelo
rescata respecto al azar. La curva de lift (derecha) es decreciente: **la ventaja relativa es máxima
en la cabeza del ranking y se diluye conforme se contacta a más gente** — en el límite, contactando
al 100 % de la cartera ambas estrategias capturan lo mismo, porque ya no hay nada que priorizar.

**Por qué importa.** Esto define dónde el modelo aporta y dónde no. Su valor **no es uniforme**:
depende críticamente de que la capacidad de contacto sea *limitada*. Si la operación pudiera
contactar a todos los leads, el modelo solo serviría para ordenar la agenda, no para aumentar
resultados.

**Qué significa comercialmente.** La herramienta debe posicionarse como un **asignador de un recurso
escaso**. Su justificación económica se refuerza —no se debilita— cuando el equipo comercial está
saturado, que es la situación habitual.

---
## 6. Practical Scenarios

Los mismos resultados, expresados como escenarios operativos. Todos los niveles de capacidad que
aparecen abajo **son puntos reales medidos** sobre el universo de evaluación —4 del experimento
Monte Carlo y 4 de la tabla de umbrales—; ninguno es extrapolado.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Qué es un «escenario» aquí</div>
<p style="margin:0;">Un escenario es simplemente <b>un tamaño de equipo</b>: «si puedo llamar a tantos leads, esto es lo que obtengo con el modelo y esto sin él». Los ocho escenarios de la tabla <b>no son proyecciones ni supuestos</b>: cada uno es un punto realmente medido sobre los datos de evaluación. Nada está extrapolado, así que ninguna cifra depende de asumir que el futuro se parecerá al pasado — solo describen lo que ya ocurrió en el conjunto evaluado.</p>
</div>

In [ ]:
tabla(pd.DataFrame({
        'Capacidad'              : res.pct_universo.map('{:.1f} % del universo'.format),
        'Leads contactados'      : res.leads.map('{:,}'.format),
        'Conv. — MODELO'         : res.conv_modelo.astype(int),
        'Conv. — AZAR'           : res.azar_media.map('{:.1f}'.format),
        'Conv./100 — modelo'     : res.conv_x100_modelo.map('{:.1f}'.format),
        'Conv./100 — azar'       : res.conv_x100_azar.map('{:.1f}'.format),
        'Diferencia'             : res.delta_conv.map('+{:.0f}'.format),
        'Lift'                   : res.lift.map('{:.2f}×'.format),
        '% del negocio capturado': (res.recall_modelo * 100).map('{:.1f} %'.format),
      }),
      titulo='Escenarios de capacidad comercial',
      destacar=['Conv. — MODELO', 'Lift'],
      nota='Guardada en <code>tablas_montecarlo/escenarios_capacidad.csv</code>.')

esc = pd.DataFrame({'capacidad_pct': res.pct_universo, 'leads': res.leads,
                    'conv_modelo': res.conv_modelo, 'conv_azar': res.azar_media,
                    'lift': res.lift, 'recall': res.recall_modelo})
esc.to_csv(TAB_DIR / 'escenarios_capacidad.csv', index=False)

ind = []
for cont in (100, 250, 500):
    for _, r in comp.iterrows():
        if r.pct_universo > 20: continue
        ind.append({'Contactos del asesor': f'{cont:,}',
                    'Tramo del ranking asignado': f'Top {r.pct_universo:.0f}%',
                    'Conv. esperadas — MODELO': f'{cont * r.precision_modelo:.1f}',
                    'Conv. esperadas — AZAR'  : f'{cont * TASA_BASE:.1f}',
                    'Conv. adicionales'       : f'+{cont * (r.precision_modelo - TASA_BASE):.1f}'})
tabla(pd.DataFrame(ind),
      titulo='Carga de trabajo individual — resultado esperado de un asesor',
      destacar=['Conv. adicionales'],
      nota='Se aplica la precisión <b>media</b> del tramo: dentro de cada tramo los leads mejor '
           'rankeados convierten más que los del borde inferior, así que el reparto interno no es '
           'uniforme. Ejemplo de lectura: si a un asesor se le asignan 250 leads tomados del Top 10 % '
           f'del ranking, se espera que convierta ~{250*r10.precision_modelo:.0f} frente a '
           f'~{250*TASA_BASE:.0f} si esos 250 leads se hubieran tomado al azar.')

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer las dos tablas de escenarios</div>
<p style="margin:0px 0 0;">La primera mira a <b>todo el equipo</b>; la segunda, a <b>una sola persona</b>.</p>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 1 — Escenarios de capacidad comercial</p>
<p style="margin:4px 0 0;">Son los ocho niveles de la tabla maestra, reordenados para lectura de negocio. Cada fila sigue siendo «si puedo hacer <i>tantas</i> llamadas, esto obtengo».</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Capacidad / Leads contactados</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El tamaño de la operación, en porcentaje de la cartera y en llamadas.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv. — MODELO / AZAR</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ventas capturadas con y sin priorización.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv./100 — modelo / azar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La productividad de cada 100 llamadas en ambos casos.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Diferencia</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ventas adicionales que aporta el modelo en ese escenario.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lift</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas veces mejor que el azar.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>% del negocio capturado</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Qué parte de las 739 ventas del universo se lleva. <b>Es la columna que responde «¿cuánto me estoy dejando?».</b></td></tr>
</table>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 2 — Carga de trabajo individual</p>
<p style="margin:4px 0 0;">Baja los mismos números a la escala de un asesor. Las filas combinan <b>tres cargas de trabajo</b> (100, 250 y 500 contactos) con <b>tres tramos del ranking</b> de los que se le pueden asignar leads.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Contactos del asesor</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas llamadas hace esa persona en el periodo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Tramo del ranking asignado</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">De qué parte de la lista salen sus leads.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv. esperadas — MODELO</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ventas que cabe esperar si sus leads salen de ese tramo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv. esperadas — AZAR</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Ventas que cabe esperar si sus leads se toman sin criterio. Es siempre el 3,26 % de sus contactos.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Conv. adicionales</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">La diferencia, para una sola persona.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Un matiz importante:</b> se aplica la precisión <b>media</b> del tramo. Dentro de un mismo tramo los leads de arriba convierten más que los del borde, así que dos asesores con leads del mismo tramo no obtendrán exactamente lo mismo. La tabla da el valor esperado, no una garantía individual.</p>
</div>

In [ ]:
display(Markdown(f"""
### Cómo leer los escenarios

**Capacidad ajustada — {int(r5.leads):,} leads ({r5.pct_universo:.0f} % del universo).** Es el caso
de un equipo pequeño o de una campaña corta. El modelo captura **{int(r5.conv_modelo)} conversiones
frente a {r5.azar_media:.0f}** del azar: **{r5.lift:.1f}× más**. Con {r5.conv_x100_modelo:.1f}
conversiones por cada 100 contactos, casi **1 de cada 5 contactos priorizados** produce resultado,
contra 1 de cada {100/r5.conv_x100_azar:.0f} al azar. Es el escenario donde la priorización es más
decisiva.

**Capacidad media — {int(r10.leads):,} leads ({r10.pct_universo:.0f} %).** El modelo captura el
**{r10.recall_modelo*100:.0f} % de todas las conversiones del universo contactando solo al 10 % de la
cartera**. Diferencia: **+{r10.delta_conv:.0f} conversiones**.

**Punto operativo del proyecto — {int(rop.leads):,} leads ({rop.pct_universo:.0f} %).** Es el punto
que el proyecto seleccionó: el menor umbral que alcanza recall ≥ 70 %. Captura
**{int(rop.conv_modelo)} de {N_CONV} conversiones ({rop.recall_modelo*100:.0f} %)** frente a
**{rop.azar_media:.0f}** del azar. El lift baja a **{rop.lift:.2f}×** porque se contacta a casi un
tercio de la cartera, pero es el punto que **maximiza la cobertura del negocio** manteniendo una
ventaja clara.

La elección entre el Top 10 % y este punto es una **decisión de negocio, no estadística**: depende de
si la prioridad es *eficiencia por contacto* (Top 5–10 %) o *cobertura total del negocio*
({rop.pct_universo:.0f} %).
"""))

nota('<b>Qué NO se puede derivar de estos resultados.</b> No hay datos sobre costo por contacto, '
     'margen por venta ni duración del ciclo comercial, de modo que <b>no se calcula ROI monetario</b>. '
     f'Convertir «{r10.delta_conv:.0f} conversiones adicionales» a valor económico requiere el ticket '
     'promedio, que está fuera del alcance de este experimento.', 'alerta')

---
## 7. Risk and Uncertainty

Toda la evidencia anterior proviene de **un** conjunto de evaluación. Esta sección examina qué tan
robusta es la ventaja y qué riesgos existen al proyectarla hacia la operación futura.

### 7.1 Variabilidad del azar y del propio modelo

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Qué se está poniendo a prueba en esta sección</div>
<p style="margin:0 0 8px;">Hasta aquí la evidencia viene de <b>una sola</b> partición de los datos. La objeción legítima es: «¿y si tuvieron suerte con esa partición?». Para responderla se cambia una cosa cada vez y se comprueba si la ventaja sobrevive:</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;"><b>Otra partición (otra semilla de validación cruzada):</b> se reparten los datos de otra forma entre entrenamiento y evaluación, y se mira si las métricas se derrumban.</li>
<li style="margin-bottom:6px;"><b>Otros 32 modelos:</b> 4 algoritmos × 4 conjuntos de variables × 2 tratamientos del desbalance. Si la ventaja apareciera solo con el modelo elegido, sería sospechosa; si aparece en todas las combinaciones, entonces lo que funciona es <b>priorizar con información</b>, no un modelo concreto.</li>
<li style="margin-bottom:6px;"><b>Otro universo de leads:</b> se repite el ejercicio sobre un conjunto distinto de datos.</li>
</ul><p style="margin:12px 0 0;">Vocabulario: la <b>validación cruzada</b> consiste en entrenar y evaluar el modelo varias veces sobre trozos distintos de los datos, de modo que ningún resultado se mida sobre los mismos casos con los que se aprendió. El <b>PR-AUC</b> y el <b>ROC-AUC</b> son notas globales de 0 a 1 sobre qué tan bien el modelo ordena a los leads; con clases tan desbalanceadas, el PR-AUC es el más informativo de los dos.</p>
</div>

In [ ]:
tabla(pd.DataFrame({
        'Corte'                      : [f'Top {p:.0f}%' for p in comp.pct_universo],
        'Azar: peor 5 %'             : [f'≤ {np.percentile(dist[int(t)], 5):.0f}' for t in comp.leads],
        'Azar: mejor 5 %'            : [f'≥ {np.percentile(dist[int(t)], 95):.0f}' for t in comp.leads],
        'Azar: mejor caso en 200k'   : comp.azar_max.astype(int),
        'Modelo'                     : comp.conv_modelo.astype(int),
        'Mejor azar / modelo'        : (comp.azar_max / comp.conv_modelo * 100).map('{:.0f} %'.format),
        'Simulaciones que lo igualan': [int((dist[int(t)] >= m).sum())
                                        for t, m in zip(comp.leads, comp.conv_modelo)],
      }),
      titulo='¿Qué tan variable es el desempeño del azar?', destacar=['Modelo'])

tabla(pd.DataFrame({
        'Semilla de CV'  : metricas.CV,
        'PR-AUC'         : metricas.pr_auc.map('{:.4f}'.format),
        'ROC-AUC'        : metricas.roc_auc.map('{:.4f}'.format),
        'Lift @10 %'     : metricas['lift@10%'].map('{:.2f}×'.format),
        'Recall @20 %'   : metricas['recall@20%'].map('{:.3f}'.format),
        '% leads r≥70 %' : metricas.pct_leads_r70.map('{:.1f} %'.format),
        'Precisión r≥70 %': metricas.prec_r70.map('{:.4f}'.format),
      }),
      titulo='Variabilidad del modelo, no solo del azar — dos semillas de validación cruzada',
      nota='Fuente: <code>modelo_final_metricas.csv</code> (§21.4 del notebook final).')

rec_sel, rec_conf   = metricas.loc[0, 'recall@20%'], metricas.loc[1, 'recall@20%']
conv_sel, conv_conf = rec_sel * N_CONV, rec_conf * N_CONV

nota(f'En el Top 20 %, el recall varía entre <b>{rec_conf:.3f} y {rec_sel:.3f}</b> según la semilla de '
     f'validación cruzada, lo que equivale a <b>{conv_conf:.0f}–{conv_sel:.0f} conversiones</b> '
     f'capturadas. Incluso en el escenario menos favorable ({conv_conf:.0f} conv.), el modelo queda '
     f'<b>{conv_conf - r20.azar_media:.0f} conversiones por encima de la media del azar</b> '
     f'({r20.azar_media:.0f}) y muy por encima del techo de su IC 95 % ({r20.azar_p97_5:.0f}). '
     f'<b>La ventaja no depende de la partición.</b>', 'ok')

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer las dos tablas de riesgo</div>
<p style="margin:0px 0 0;">Hasta aquí se comparó el modelo con el azar. Estas dos tablas preguntan algo distinto: <b>¿cuánto puede variar cada uno de los dos?</b></p>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 1 — ¿Qué tan variable es el desempeño del azar?</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar: peor 5 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El valor por debajo del cual queda el 5 % de los sorteos: un mal día del azar.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar: mejor 5 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El valor que solo supera el 5 % de los sorteos: un <b>buen</b> día del azar.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Azar: mejor caso en 200k</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El techo absoluto de la suerte en 200 000 intentos.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Mejor azar / modelo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Qué porcentaje del resultado del modelo alcanza ese mejor caso. En Top 10 % llega apenas al <b>36 %</b>: <b>ni con su mejor racha el azar consigue la mitad</b> de lo que consigue el modelo.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Simulaciones que lo igualan</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas de las 200 000 llegaron al resultado del modelo. Cero en los cuatro cortes.</td></tr>
</table>
<p style="margin:18px 0 0; font-weight:700; color:#7B1FA2;">Tabla 2 — Dos semillas de validación cruzada</p>
<p style="margin:4px 0 0;">Aquí lo que varía es <b>el modelo</b>, no el azar. La «semilla» decide cómo se reparten los datos entre entrenamiento y evaluación; cambiarla equivale a <b>rehacer el estudio con otro reparto</b>. Si los resultados se derrumbaran, la ventaja habría sido suerte de la partición.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Semilla de CV</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Las dos particiones probadas: <i>selección</i> (con la que se eligió el modelo) y <i>confirmación</i> (una segunda, independiente).</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>PR-AUC</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Nota global de 0 a 1 sobre qué tan bien ordena el modelo. <b>Es la métrica más informativa aquí</b>, porque está pensada para casos con muy pocos positivos.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>ROC-AUC</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Otra nota global de 0 a 1. Con clases tan desbalanceadas tiende a verse optimista, por eso se reporta junto a la anterior y no sola.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lift @10 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántas veces mejor que el azar en el Top 10 %.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Recall @20 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Qué fracción de las ventas captura contactando al 20 % de la cartera.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>% leads r≥70 % / Precisión r≥70 %</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuánta cartera exige el punto operativo y con qué precisión trabaja en él.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Qué mirar:</b> que las dos filas se parezcan. Se parecen mucho (PR-AUC 0,2215 frente a 0,2193). La única diferencia apreciable está en el recall@20 %, y aun en el caso menos favorable el modelo sigue muy por encima del techo del azar.</p>
</div>

### 7.2 Réplica independiente sobre 32 configuraciones de modelo

In [ ]:
op  = pd.read_csv(RESULTS_DIR / 'operational_eval_conversion.csv')
rob = op.groupby('threshold_pct').agg(
        configuraciones=('sig_vs_mc', 'size'), superan_IC95=('sig_vs_mc', 'sum'),
        lift_min=('lift_vs_mc', 'min'), lift_mediana=('lift_vs_mc', 'median'),
        lift_max=('lift_vs_mc', 'max'), leads=('n_selected', 'first')).reset_index()

tabla(pd.DataFrame({
        'Corte'                  : rob.threshold_pct.map('Top {:.0f}%'.format),
        'Leads'                  : rob.leads.map('{:,}'.format),
        'Configuraciones'        : rob.configuraciones,
        'Superan el IC 95 % del azar': rob.superan_IC95.astype(str) + ' / ' + rob.configuraciones.astype(str),
        'Lift mínimo'            : rob.lift_min.map('{:.2f}×'.format),
        'Lift mediano'           : rob.lift_mediana.map('{:.2f}×'.format),
        'Lift máximo'            : rob.lift_max.map('{:.2f}×'.format),
      }),
      titulo='32 configuraciones de modelo — 4 algoritmos × 4 conjuntos de variables × 2 tratamientos de desbalance',
      destacar=['Superan el IC 95 % del azar'],
      nota='Fuente: <code>operational_eval_conversion.csv</code>, Monte Carlo de 10 000 iteraciones. '
           '<b>Atención:</b> corresponde al universo exploratorio anterior (n≈8 480, tasa base ≈1,74 %), '
           '<b>no al modelo final</b>. Se incluye como evidencia independiente de robustez, no como '
           'medición del modelo final.')

fig, ax = plt.subplots(figsize=(10.5, 5.2))
for i, thr in enumerate(sorted(op.threshold_pct.unique())):
    v   = op[op.threshold_pct == thr]['lift_vs_mc'].values
    jit = np.random.default_rng(1).normal(0, 0.07, len(v))
    ax.scatter(v, np.full(len(v), i) + jit, s=42, color=C_MODELO, alpha=0.55,
               edgecolor='white', linewidth=0.8,
               label='Configuraciones de modelo (32)' if i == 0 else None)
    ax.scatter([np.median(v)], [i], s=150, marker='|', color=INK, linewidth=2.2,
               label='Mediana' if i == 0 else None)
ax.axvline(1, color=C_AZAR, lw=2.2, ls='--', label='Selección aleatoria (lift = 1)')
ax.set_yticks(range(4)); ax.set_yticklabels([f'Top {int(t)}%' for t in sorted(op.threshold_pct.unique())])
ax.set_xlabel('Lift frente a la media Monte Carlo (veces mejor que el azar)', fontweight='bold')
ax.set_title('Robustez: la ventaja no depende de una configuración afortunada\n'
             '32 configuraciones de modelo, universo exploratorio anterior — todas superan al azar',
             fontweight='bold', fontsize=12)
ax.legend(loc='lower right'); ax.set_xlim(0, 4.9); ax.grid(axis='y', alpha=0); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(FIG_DIR / 'mc_robustez_configuraciones.png', bbox_inches='tight'); plt.show()

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Cómo leer la tabla de las 32 configuraciones</div>
<p style="margin:0px 0 0;">Esta es la prueba más exigente del reporte: en lugar de defender <i>un</i> modelo, se construyeron <b>32 modelos distintos</b> —4 algoritmos × 4 conjuntos de variables × 2 formas de tratar el desbalance— y se comprobó si <b>todos</b> le ganan al azar. Si la ventaja apareciera solo con el modelo elegido, sería sospechosa.</p>
<table style="border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 0;">
<tr><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Columna</th><th style="text-align:left; padding:7px 10px; border-bottom:2px solid #D9C2E8; color:#7B1FA2; font-weight:700;">Qué significa</th></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Corte</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El nivel de capacidad evaluado, igual que en el resto del reporte.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Leads</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántos leads se contactan en ese corte. <b>Son números mucho más pequeños</b> (424, 848…) porque este ejercicio se hizo sobre un universo anterior y más chico.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Configuraciones</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">Cuántos modelos se pusieron a prueba en esa fila: los 32.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Superan el IC 95 % del azar</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>La columna clave.</b> Cuántos de esos 32 quedaron por encima del techo del rango habitual del azar. Dice <b>32 / 32</b> en los cuatro cortes.</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lift mínimo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El <b>peor</b> de los 32 modelos. Es el dato más importante de la fila: incluso el peor gana (1,32× en el corte más exigente).</td></tr>
<tr><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;"><b>Lift mediano / máximo</b></td><td style="padding:7px 10px; border-bottom:1px solid #EADCF3; vertical-align:top;">El modelo típico y el mejor de los 32.</td></tr>
</table>
<p style="margin:14px 0 0;"><b>Qué demuestra:</b> que lo que funciona no es un algoritmo concreto ni una selección afortunada de variables, sino <b>priorizar con información</b>. Cualquier modelo razonable entrenado sobre estos datos le gana a llamar sin criterio.</p>
<p style="margin:10px 0 0;"><b>Y qué no demuestra —importante para no sobreinterpretar:</b> estos 32 modelos se evaluaron sobre el <b>universo exploratorio anterior</b> (unos 8 480 leads, tasa base 1,74 %), no sobre el universo del modelo final. Por eso los lifts no son comparables cifra a cifra con el resto del reporte: valen como <b>evidencia independiente de robustez</b>, no como medición del modelo final.</p>
</div>

In [ ]:
display(Markdown(f"""
### 7.3 Lectura de los riesgos

**¿Podría el azar acercarse al modelo alguna vez?** En este conjunto de evaluación, no. En
**{N_MC_HD:,} simulaciones por corte**, el mejor resultado aleatorio alcanzó como máximo el
**{r10.azar_max / r10.conv_modelo * 100:.0f} %** del resultado del modelo en el Top 10 %. La
frecuencia con que el azar iguala al modelo es **cero en la simulación** y del orden de
**{r10.p_azar_ge_modelo:.0e}** en el cálculo exacto.

**¿Qué tan robusta es la ventaja?** Se sostiene bajo tres pruebas independientes:

1. **Cambio de partición** — con la segunda semilla de CV el recall@20 % baja de {rec_sel:.3f} a
   {rec_conf:.3f}, unas {conv_sel-conv_conf:.0f} conversiones, y la ventaja sobre el azar apenas se mueve.
2. **Cambio de algoritmo y de variables** — **{int(rob.superan_IC95.sum())} de
   {int(rob.configuraciones.sum())}** combinaciones probadas superan el techo del IC 95 % del azar.
   La ventaja es una propiedad de *priorizar con información*, no de un modelo concreto.
3. **Cambio de universo** — la réplica sobre un universo distinto (n≈8 480, tasa base 1,74 %)
   reproduce lifts de 2,7×–4,3× en Top 5 %, del mismo orden que el modelo final.

**¿Qué implica el número de conversiones disponible?** El universo contiene solo **{N_CONV}
conversiones**. En el Top 5 % el modelo acierta {int(r5.conv_modelo)} de ellas: cifras de tres
dígitos, suficientes para que las estimaciones sean estables. Pero cualquier análisis más fino —por
proyecto, canal o segmento— **fragmenta esas {N_CONV} conversiones**, y ahí el error de estimación
crece rápidamente. Este experimento **no autoriza a afirmar que el lift sea el mismo en cada
proyecto**.

### El riesgo principal: la diferencia entre dos enunciados

| Lo que el experimento demuestra | Lo que NO demuestra |
|---|---|
| El modelo superó al azar **en estos datos históricos**, de forma masiva y consistente | Que el modelo obtendrá **este mismo lift en producción** |
| El ordenamiento captura señal real, no ruido de partición | Que la señal se mantendrá si cambia el mix de canales, la oferta o el mercado |
| La ventaja es estable ante cambios de algoritmo, variables y partición | Que sea estable ante cambios de **período** — no hay validación temporal aquí |

**Riesgos concretos al generalizar:**

- **Deriva temporal.** El modelo se evaluó sobre leads históricos ya resueltos. Un cambio en la
  mezcla de canales, en los precios o en el contexto macro degrada el ranking sin previo aviso.
- **Efecto de la propia intervención.** Al priorizar, la operación **deja de contactar** el fondo del
  ranking y dejará de observar conversiones ahí. El histórico futuro quedará contaminado por la
  propia decisión del modelo y no permitirá volver a medir esta comparación de la misma manera.
  Conviene reservar una fracción de contactos aleatorios como grupo de control.
- **Evaluación fuera de muestra, pero no fuera de tiempo.** Las probabilidades son *out-of-fold*
  —cada lead se puntúa con un modelo que no lo vio en entrenamiento—, lo cual descarta el
  sobreajuste directo, pero no sustituye a una validación sobre un período posterior.
"""))

---
## 8. Qué demuestra realmente el experimento Monte Carlo

<div style="border-left:5px solid #1565C0; background:#F5F9FF; padding:16px 20px; border-radius:4px; font-size:14px; color:#263238; line-height:1.65;">
<b style="color:#0D47A1;">La afirmación central</b><br><br>
El experimento <b>no intenta demostrar que el modelo predice perfectamente quién comprará</b>.
Evalúa si el modelo permite <b>asignar un recurso comercial limitado de manera más eficiente que
una estrategia sin priorización</b>.
</div>

<br>

Son dos preguntas muy distintas, y confundirlas lleva a evaluar el modelo con la vara equivocada.

### Por qué la comparación correcta es "modelo vs. azar" y no "modelo vs. perfección"

| | Pregunta de predicción | Pregunta de asignación *(la de este experimento)* |
|---|---|---|
| **Formulación** | ¿Este lead va a comprar, sí o no? | Dado que solo puedo llamar a N leads, ¿a cuáles llamo primero? |
| **Métrica natural** | Accuracy, precisión, F1 | Recall@K, lift, conversiones capturadas a capacidad fija |
| **Referencia justa** | Un clasificador perfecto | **Lo que haría la operación sin el modelo: elegir sin criterio** |
| **Veredicto sobre este modelo** | Insuficiente: la precisión en el Top 10 % es ~13 % | Contundente: captura 4,2× más conversiones con el mismo esfuerzo |

Con una tasa base de 3,26 %, **ningún modelo podría alcanzar alta precisión**: la información
disponible sobre un lead no determina la compra, solo la hace más o menos probable. Exigirle al
modelo que "acierte quién compra" es exigirle que resuelva un problema que los datos no contienen.

### Por qué el azar es la referencia relevante para el negocio

La alternativa real a usar el modelo **no es un oráculo**: es el proceso actual, en el que los leads
se trabajan por orden de llegada, por disponibilidad del asesor o por criterios informales. La
selección aleatoria es la **formalización estadística de "sin criterio de priorización"**, y por eso
es el punto de comparación honesto. La pregunta que responde el Monte Carlo es exactamente la que se
hace un gerente comercial:

> *Si mañana ordeno la lista con el modelo en lugar de trabajarla como siempre,
> ¿cuánto negocio adicional entra con el mismo equipo?*

### El balance final

<table>
<tr>
<td width="50%" valign="top" style="background:#F4FBF5; border-left:5px solid #2E7D32; padding:14px 18px;">
<b style="color:#2E7D32;">Lo que SÍ demuestra</b>
<ul style="margin:8px 0 0 0; padding-left:18px; line-height:1.7;">
<li>Que el ordenamiento del modelo <b>contiene información real</b> sobre la propensión a convertir,
en un grado que el azar no puede replicar (P ≈ 10⁻¹¹⁷).</li>
<li>Que esa información <b>se traduce en más conversiones capturadas a capacidad constante</b>.</li>
<li>Que el efecto <b>no depende</b> del algoritmo, del conjunto de variables ni de la partición.</li>
</ul>
</td>
<td width="50%" valign="top" style="background:#FFF8F0; border-left:5px solid #F57C00; padding:14px 18px;">
<b style="color:#E65100;">Lo que NO demuestra</b>
<ul style="margin:8px 0 0 0; padding-left:18px; line-height:1.7;">
<li>Que el modelo identifique correctamente a <b>cada comprador individual</b>.</li>
<li>Que el desempeño se mantendrá en un <b>período futuro</b> — no hay validación temporal.</li>
<li>Que el <b>beneficio económico neto</b> sea positivo — no se modelaron costos ni márgenes.</li>
<li>Que contactar a un lead priorizado <b>cause</b> la conversión: mide capacidad de ordenar, no
efecto causal.</li>
</ul>
</td>
</tr>
</table>

---
## 9. Limitations

Limitaciones observadas en estos resultados concretos, no una lista genérica.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">Por qué las limitaciones no contradicen los resultados</div>
<p style="margin:0 0 8px;">Un punto que suele confundir: el modelo <b>acierta poco cuando se le pide adivinar quién compra</b>, y aun así <b>ordena muy bien la lista de llamadas</b>. Las dos cosas son ciertas a la vez.</p><ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;"><b>Desbalance de clases:</b> solo 3 de cada 100 leads compran. Con esa proporción, ningún modelo puede tener alta precisión — pero sí puede subir mucho la concentración de compradores en la cabeza de la lista, que es lo que se usa.</li>
<li style="margin-bottom:6px;"><b>Por qué la <i>accuracy</i> engaña:</b> decir «nadie va a comprar» acierta el 96,7 % de las veces y no sirve absolutamente para nada. Por eso el reporte se apoya en recall y lift, no en exactitud.</li>
<li style="margin-bottom:6px;"><b>Alcance:</b> los resultados valen para los leads con las cinco variables informadas y para el periodo evaluado. Los leads con datos incompletos quedan fuera, y el lift no está validado por subgrupos (proyecto, canal, segmento) porque 739 conversiones repartidas se vuelven demasiado pocas para concluir nada.</li>
</ul>
</div>

In [ ]:
display(Markdown(f"""
**1. Desbalance extremo de clases, y su consecuencia directa.** La tasa base es
**{TASA_BASE*100:.2f} %** ({N_CONV} conversiones en {N_UNIVERSO:,} leads). Esto fija un techo a la
precisión alcanzable y explica por qué, incluso en el Top 5 %, la precisión es de
**{r5.precision_modelo*100:.1f} %**. Métricas como *accuracy* (0,7222) son engañosas aquí: predecir
"nadie convierte" daría {(1-TASA_BASE)*100:.1f} % de exactitud. Por eso el análisis se apoya en
recall@K y lift, no en exactitud.

**2. El tamaño de la muestra positiva limita el detalle del análisis.** Con **{N_CONV} conversiones**
en total, los cortes altos son sólidos ({int(r5.conv_modelo)}–{int(rop.conv_modelo)} positivos
capturados), pero cualquier desagregación —por proyecto, canal, rango de ingreso o segmento— reparte
esos {N_CONV} casos y vuelve las estimaciones inestables. **Este experimento no valida el lift a
nivel de subgrupo.**

**3. Dependencia del conjunto de evaluación.** Los resultados corresponden al **universo H**: leads
con las cinco variables requeridas informadas ({N_UNIVERSO:,} de los ~42 400 leads del histórico
completo). Los leads con datos incompletos **quedan fuera del alcance del modelo** y no aparecen en
esta comparación. En producción habrá que decidir explícitamente qué hacer con ellos, y esa decisión
afecta el desempeño agregado de la operación, que será inferior al medido aquí sobre un universo
depurado.

**4. Sesgo del proceso comercial histórico.** La variable objetivo registra qué leads **efectivamente
convirtieron bajo el proceso comercial existente**. Si históricamente ciertos leads recibieron mejor
atención —por canal, por asesor, por momento— el modelo aprende parcialmente **el comportamiento del
equipo de ventas**, no solo la propensión intrínseca del cliente. Un lead "bueno" puede serlo porque
se le atendió bien. Esto infla el desempeño medido y es imposible de separar sin experimentación
aleatorizada.

**5. La comparación es out-of-fold, no fuera de tiempo.** Cada lead se puntúa con un modelo que no lo
vio en entrenamiento, lo que descarta el sobreajuste directo. Pero los folds mezclan períodos: **el
modelo puede haber aprendido de leads futuros para predecir leads pasados**. Una validación con corte
temporal daría una estimación más conservadora y más parecida a la operación real.

**6. La simulación aleatoria no captura la incertidumbre del modelo.** El Monte Carlo aleatoriza
**únicamente la selección de leads**; el resultado del modelo entra como un número fijo. El intervalo
de confianza reportado es el del *azar*, no el del modelo. La variabilidad del modelo se estimó por
separado (dos semillas de CV: recall@20 % de {rec_conf:.3f} a {rec_sel:.3f}) y no altera las
conclusiones, pero **no existe un intervalo de confianza formal para el desempeño del modelo** en
este experimento.

**7. Coexistencia de artefactos de distintas corridas.** En `qualified_run/results/` conviven salidas
del 15 de julio (universo de 20 304 leads / 625 conversiones) y del 4 de agosto (22 656 / 739). Los
archivos `modelo_final_metricas_topk.csv`, `..._ranking.csv` y `..._umbral.csv` pertenecen a la
corrida antigua y **contradicen** a los del modelo final si se leen juntos. Este notebook los excluye
deliberadamente, pero es un riesgo real de interpretación errónea para cualquier otro lector de esa
carpeta.

**8. Ausencia de variables económicas.** No hay costo por contacto, margen por venta ni valor del
ciclo. Todas las conclusiones están en **unidades de conversión y de esfuerzo**, no en dinero. La
afirmación "el modelo es rentable" **no se ha demostrado aquí**; lo demostrado es que es *más
eficiente a esfuerzo constante*.
"""))

---
## 10. Conclusions

Respuestas directas a las ocho preguntas que motivaron el análisis.

<div style="border-left:5px solid #7B1FA2; background:#FBF6FE; padding:16px 20px; border-radius:4px; margin:14px 0; max-width:940px; color:#3B2A46; font-size:13.5px; line-height:1.65;">
<div style="font-size:10.5px; font-weight:700; letter-spacing:.09em; text-transform:uppercase; color:#7B1FA2; margin-bottom:6px;">En palabras simples</div>
<div style="font-size:15px; font-weight:700; color:#7B1FA2; margin-bottom:10px;">El resumen de todo, en cinco líneas</div>
<ul style="margin:8px 0 0; padding-left:20px;">
<li style="margin-bottom:6px;">De cada 100 leads, solo unos 3 compran. Nadie puede llamarlos a todos.</li>
<li style="margin-bottom:6px;">El modelo no adivina quién compra: <b>ordena la lista</b> para que los compradores queden arriba.</li>
<li style="margin-bottom:6px;">Con las mismas llamadas, ese orden captura <b>varias veces más ventas</b> que llamar sin criterio.</li>
<li style="margin-bottom:6px;">La diferencia es tan grande que <b>el azar no puede alcanzarla ni con suerte extrema</b>, y se repite al cambiar la partición, el algoritmo y el universo de datos.</li>
<li style="margin-bottom:6px;">Su valor depende de que la capacidad sea limitada: <b>cuanto más saturado está el equipo, más aporta priorizar</b>.</li>
</ul>
</div>

In [ ]:
display(Markdown(f"""
**1. ¿El modelo supera al azar?**
Sí, sin ambigüedad, en los ocho niveles de capacidad evaluados. En el Top 10 % captura
**{int(r10.conv_modelo)} conversiones frente a {r10.azar_media:.0f}** de la selección aleatoria.

**2. ¿Por cuánto?**
Entre **{res.lift.min():.2f}× y {res.lift.max():.2f}×** según la capacidad: **{r5.lift:.2f}×** en
Top 5 %, **{r10.lift:.2f}×** en Top 10 %, **{r20.lift:.2f}×** en Top 20 % y **{r30.lift:.2f}×** en
Top 30 %. En términos absolutos, entre **+{comp.delta_conv.min():.0f} y
+{comp.delta_conv.max():.0f} conversiones** por ciclo.

**3. ¿Qué tan consistente es esa ventaja?**
Muy consistente. Se mantiene (a) en los cuatro cortes del experimento y en los cuatro umbrales
operativos adicionales; (b) con las dos semillas de validación cruzada; (c) en
**{int(rob.superan_IC95.sum())} de {int(rob.configuraciones.sum())}** combinaciones de algoritmo,
variables y tratamiento de desbalance; y (d) en un universo de evaluación distinto. **No es el
resultado de una configuración afortunada.**

**4. ¿Qué probabilidad tiene una selección aleatoria de igualar o superar al modelo?**
Esencialmente nula: **{r10.p_azar_ge_modelo:.1e}** en el Top 10 % (cálculo exacto). En
**{N_MC_HD:,} simulaciones** por corte, **ninguna** alcanzó al modelo; el mejor resultado aleatorio
en el Top 10 % fue de **{int(r10.azar_max)} conversiones** contra **{int(r10.conv_modelo)}**. El
modelo está a **{r10.z_modelo:.0f} desviaciones estándar** de la media del azar.

**5. ¿Qué significa esto comercialmente?**
Que **hay negocio recuperable dentro de la cartera actual sin gastar más**. Con capacidad del 10 %,
trabajar sin priorización cuesta **{r10.delta_conv:.0f} conversiones por ciclo**. La ganancia no
proviene de contactar más leads ni de mejorar el discurso de venta, sino de **cambiar el orden de la
lista**: la intervención más barata disponible.

**6. ¿Qué beneficio obtiene un asesor?**
Su tasa de éxito pasa de **{r10.conv_x100_azar:.1f} a {r10.conv_x100_modelo:.1f} conversiones por
cada 100 contactos**: de una venta cada ~{r10.contactos_x_conv_azar:.0f} contactos a una cada
~{r10.contactos_x_conv_modelo:.0f}. Con la misma jornada, **{r10.lift:.1f}× más resultados**. Con
honestidad: la mayoría de sus contactos seguirá sin convertir — el modelo mejora la eficiencia, no
elimina el rechazo.

**7. ¿Qué beneficio obtiene la empresa?**
Con el punto operativo del proyecto ({int(rop.leads):,} leads, {rop.pct_universo:.0f} % de la
cartera) **captura el {rop.recall_modelo*100:.0f} % de todo el negocio disponible contactando menos
de un tercio de los leads**. Alternativamente, puede sostener el resultado actual con
**{esf.iloc[1]['Reducción de esfuerzo']}** de esfuerzo comercial y reasignar la capacidad liberada.
Además obtiene un criterio explícito, auditable y reproducible para asignar el recurso más caro de la
operación: el tiempo del equipo de ventas.

**8. ¿Los resultados justifican utilizar el modelo como herramienta de priorización?**
**Sí.** La evidencia es estadísticamente incontestable y comercialmente material: un
**+{r10.incremento_pct:.0f} %** de conversiones a costo operativo constante. La justificación es
sólida **para el uso propuesto —ordenar la cola de contacto—** y no para otros: el modelo no debe
emplearse para descartar leads de forma definitiva, para prometer una tasa de cierre a un asesor, ni
para decisiones sobre clientes individuales. Su valor está demostrado como *asignador de un recurso
escaso*, y así debe implantarse.
"""))

nota('<b>Condiciones recomendadas para la implantación</b>, que se desprenden de las limitaciones: '
     'mantener un <b>grupo de control aleatorio</b> (5–10 % de los contactos) para poder seguir '
     'midiendo el lift en producción; <b>revalidar el modelo con corte temporal</b> tras el primer '
     'ciclo completo; y definir explícitamente el <b>tratamiento de los leads fuera del universo H</b> '
     '(datos incompletos), que el modelo no puede puntuar.', 'info')

---
### Archivos generados por este notebook

In [ ]:
figs = sorted(FIG_DIR.glob('*.png')); tabs = sorted(TAB_DIR.glob('*.csv'))
display(Markdown(
    '**Figuras** — `04 Notebook Final/figuras_montecarlo/`\n\n'
    + '\n'.join(f'- `{f.name}`' for f in figs)
    + '\n\n**Tablas** — `04 Notebook Final/tablas_montecarlo/`\n\n'
    + '\n'.join(f'- `{f.name}`' for f in tabs)))

nota('Ningún archivo del experimento original fue modificado. Este notebook solo lee de '
     '<code>03 Prediccion/qualified_run/results/</code> y escribe en sus dos carpetas propias.', 'ok')